# G9 — Fractal-trained vision models vs the eight encoders (Spearman shape agreement)

**Question.** PRH's hypothesis (report §2.3) says *well-trained* models converge because they
recover the statistics of one underlying reality. Experiment A's random-weights control shows
the similarity is **learned**. It does not show it is learned **from reality**: any training on
any structured data might produce it. A model trained only on fractals has training without
reality. Where does it land?

**Two fractal model types, matched data (FractalDB-1k: 1M rendered fractals, 1,000 formula-defined
classes, zero natural images):**

| space | architecture | trained on | features |
|---|---|---|---|
| `frac_cnn` | ResNet-50 | FractalDB-1k (Kataoka et al., ACCV 2020) | 2048-d global average pool |
| `frac_vit` | DeiT (ViT, patch 16) | FractalDB-1k (Nakashima et al., 2021) | cls+patch, as the DINOv2 caches |

**Brackets — the controls that make the number interpretable.** Each fractal model gets two
architecture-identical twins: **random weights** (architecture only) and **ImageNet-1k** (architecture
+ reality). A 32×32 **pixel** baseline is the floor.

**Statistic.** Exactly the E.12 / Table C12m statistic: Spearman ρ between the pairwise-cosine
structures of two spaces, all 9,533 items, no map fitted. Cell 6 **reproduces the published ConvNeXt
row and bge–SBERT 0.768** before any new number is trusted.

**Reality fraction** (new): `R = (ρ_fractal − ρ_random) / (ρ_natural − ρ_random)`, per reference
encoder. 0 = fractals added nothing over architecture; 1 = fractals did as well as natural images.
Undefined when the bracket is narrower than 0.02 (same arithmetic trap as the width cliff).

### Known confounds, stated before the run

1. **P1 is the expected result.** ImageNet twins will almost certainly agree with DINOv2 more than
   fractal models do; P1 is a floor-level check. The informative quantities are **P2, P3 and the
   size of R**.
2. The ResNet-50 ImageNet twin and ConvNeXt (ImageNet-22k) share a data family, so the upper
   bracket against ConvNeXt is inflated; read R against ConvNeXt with that in mind.
3. Objective is matched (both fractal models and both twins are 1,000-class supervised
   classifiers); training recipes (epochs, augmentation) are not.
4. COCO is out of distribution for a fractal model by construction — that is the question, not a
   flaw, but it means R measures *transfer of structure to the real world*, not fractal skill.

### Pre-registered predictions — fixed before any fractal feature exists

| # | prediction | threshold | if it fails |
|---|---|---|---|
| **P1** | ρ(fractal) < ρ(ImageNet twin) | ≥7 of 8 encoders, gap > 2 combined subsample SDs, per architecture | training on *any* structured data suffices; the 'reality' reading of PRH weakens |
| **P2** | ρ(fractal) > ρ(random twin) | ≥3 of 4 image encoders, same gap rule | fractal training adds no shared structure beyond the architecture |
| **P3** | mean R(text) < mean R(image) | healthy text = bge, BERT, SBERT (GPT-2 stratified out); 95% subsample interval of R_img − R_txt excludes 0 | the language-shared structure does not especially need the world |
| **P5** | ρ(fractal) > ρ(pixels) | ≥3 of 4 image encoders, same gap rule | fractal features are no better than raw layout |
| X1 | CNN vs ViT | exploratory — no prediction; the released fractal ViT is DeiT-tiny (~5M) vs ResNet-50 (~25M), so this contrast also carries a capacity gap | — |

GPT-2 is the documented collapsed space; it is reported but never pooled into a mean (stratify
before averaging).

**Run order:** set `SYNTHETIC = True` for a 2-minute dry run of every cell on generated data, then
`False` for the real run (L4: ~15 min download + ~10 min extraction + ~10 min statistics).

## 1 — Install

In [ ]:
import importlib, importlib.util, subprocess, sys
for pkg, mod in [("timm", "timm"), ("gdown", "gdown"), ("openpyxl", "openpyxl")]:
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import timm, gdown
print("timm", timm.__version__, "| gdown", gdown.__version__)

## 2 — Storage and configuration

In [ ]:
import os, json, hashlib, time
from pathlib import Path
import numpy as np, pandas as pd

SYNTHETIC = False        # True = dry run on generated data: no downloads, no GPU, ~2 min

N_ITEMS   = 9533         # the hub corpus; every cache is truncated to this prefix (E.15)
BATCH     = 128
SUB_M     = 2000         # items per stability subsample
SUB_B     = 30           # number of subsamples
SEED      = 0
MIN_BRACKET = 0.02       # R undefined below this
GAP_K     = 2.0          # 'gap > GAP_K combined SDs'
P3_POOL   = 600          # m-out-of-n paired-bootstrap pool for the P3 R-difference CI
P3_BOOT   = 120          # item-level paired bootstrap replicates (wider but valid CI)

# optional overrides if the Drive-folder download fails -- point at the .pth files
FRACTAL_RESNET_PATH = os.environ.get("FRACTAL_RESNET_PATH", "")
FRACTAL_VIT_PATH    = os.environ.get("FRACTAL_VIT_PATH", "")
RESNET_FOLDER = "https://drive.google.com/drive/folders/1tTD-cKKEgBjacCi4ZJ6bRYOv6FsjtGt_"   # Kataoka ACCV 2020 README
VIT_FOLDER    = "https://drive.google.com/drive/folders/1r2e0Iel_DpKlW6zDI5Q_8ZfifS320W3O"   # Nakashima 2021 README

if SYNTHETIC:
    N_ITEMS, SUB_M, SUB_B = 1500, 500, 10
    P3_POOL, P3_BOOT = 300, 60
    DATA_DIR = Path("/tmp/g9_synthetic_data")
    WORK = Path("/tmp/g9_work")
else:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        base = Path("/content/drive/MyDrive")
    except Exception:
        base = Path(os.environ.get("DRIVE_ROOT", "."))
    DATA_DIR = Path(os.environ.get("DATA_DIR", base / "convergence_experiment"))
    if not DATA_DIR.exists():                      # shared folder may sit elsewhere in MyDrive
        hits = [p for p in base.glob("**/convergence_experiment") if p.is_dir()][:1]
        DATA_DIR = hits[0] if hits else DATA_DIR
    WORK = Path("/content/g9_work")

OUT = DATA_DIR / "G9_fractal"
for p in (WORK, OUT):
    p.mkdir(parents=True, exist_ok=True)
print(f"SYNTHETIC={SYNTHETIC}\nDATA_DIR={DATA_DIR} exists={DATA_DIR.exists()}\nWORK={WORK}\nOUT={OUT}")

## 3 — Core code (Spearman engine, gates, I/O, evidence table)

In [ ]:
"""G9 core: Spearman shape agreement at full n, cache discovery, identity and
reproduction gates, brackets, reality fraction, pre-registered verdicts.

Torch-free on purpose: this is the part that decides the numbers, so it is the
part tested outside Colab. Model loading and extraction live in the notebook.
"""
import fnmatch
import re
from pathlib import Path

import numpy as np
from scipy.stats import rankdata, spearmanr

# ============================================================== Spearman engine
def upper_tri_cos(X, idx=None):
    """Upper-triangle (k=1) cosine similarities in float64.

    float64 matters: the collapsed GPT-2 space has pair cosines crowded near 1,
    and at float32 resolution many of them tie exactly, which silently changes
    ordinal ranks."""
    X = np.asarray(X, dtype=np.float64)
    if idx is not None:
        X = X[idx]
    X = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-300)
    n = X.shape[0]
    v = np.empty(n * (n - 1) // 2, dtype=np.float64)
    pos, CH = 0, 1024          # row blocks: peak memory ~ CH x n, not n x n
    for s in range(0, n, CH):
        blk = X[s:s + CH] @ X.T
        for i in range(blk.shape[0]):
            r = s + i
            L = n - r - 1
            v[pos:pos + L] = blk[i, r + 1:]
            pos += L
    return v


def standardized_ranks(v, tie_tol=1e-6):
    """Ranks of v, centered and scaled to unit norm, so Spearman(a, b) = a . b.

    Ordinal ranks by argsort are exact when there are no ties. If the fraction
    of tied adjacent values exceeds tie_tol we switch to average ranks, which is
    what scipy.stats.spearmanr uses. Returns (float32 vector, tie_fraction)."""
    order = np.argsort(v, kind="stable")
    sv = v[order]
    tie_frac = float(np.mean(sv[1:] == sv[:-1])) if len(sv) > 1 else 0.0
    if tie_frac > tie_tol:
        r = rankdata(v, method="average").astype(np.float64)
    else:
        r = np.empty(len(v), dtype=np.float64)
        r[order] = np.arange(len(v), dtype=np.float64)
    r -= r.mean()
    r /= np.linalg.norm(r) + 1e-300
    return r.astype(np.float32), tie_frac


class RankBank:
    """Computes each space's standardized pair-rank vector once and reuses it.
    At n = 9,533 one vector is 45.4M float32 = 182 MB."""

    def __init__(self):
        self.r, self.ties = {}, {}

    def add(self, name, X, idx=None):
        v = upper_tri_cos(X, idx)
        self.r[name], self.ties[name] = standardized_ranks(v)
        del v
        return self.ties[name]

    def rho(self, a, b):
        # accumulate in float64: a float32 dot over 45M terms loses ~4 digits
        return float(np.dot(self.r[a].astype(np.float64), self.r[b].astype(np.float64)))


def rho_direct(X, Y, idx=None):
    """One-shot Spearman shape agreement; used on subsamples."""
    a, _ = standardized_ranks(upper_tri_cos(X, idx))
    b, _ = standardized_ranks(upper_tri_cos(Y, idx))
    return float(np.dot(a.astype(np.float64), b.astype(np.float64)))


def subsample_rhos(X, Y, m=2000, B=30, seed=0, shuffle=False):
    """rho on B item-subsamples of size m, SAME items in both spaces.

    This is a stability interval over items, not a full-n confidence interval:
    it answers 'how much would rho move on a different 2,000 images', which is
    the spread a between-model gap has to beat. shuffle=True breaks the row
    correspondence in Y and gives the null."""
    rng = np.random.default_rng(seed)
    n = X.shape[0]
    out = np.empty(B)
    for b in range(B):
        idx = rng.choice(n, m, replace=False)
        jdx = rng.permutation(idx) if shuffle else idx
        a, _ = standardized_ranks(upper_tri_cos(X, idx))
        c, _ = standardized_ranks(upper_tri_cos(Y, jdx))
        out[b] = float(np.dot(a.astype(np.float64), c.astype(np.float64)))
    return out


def paired_item_bootstrap_rho(spaces, encoders, items, B=400, seed=0):
    """Item-level paired bootstrap of shape-rho.

    spaces:   dict name -> (n, d) array (the bracket spaces AND the encoders)
    encoders: dict name -> (n, d) array to correlate against (the 8 encoders)
    items:    1-D index array defining the working pool of items (drawn ONCE upstream);
              each replicate resamples THESE indices with replacement.

    Returns rho[(space, enc)] -> np.ndarray of length B. Because every (space, enc)
    pair is evaluated on the SAME resampled item set within a replicate, any function
    of them (e.g. the reality fraction R, or an img-vs-text difference) is properly
    paired and its bootstrap distribution reflects genuine item-level uncertainty --
    NOT the spread of overlapping fixed-size subsample means (which is not a valid CI).

    This is an m-out-of-n paired bootstrap over the working pool: honest about item
    resampling, and tractable because the pool is a few thousand items, not all 9,533
    (a full-n bootstrap would rank ~45M pairs per replicate)."""
    rng = np.random.default_rng(seed)
    items = np.asarray(items)
    m = items.size
    # cache each space's cosine vector and rank on the FIXED pool once is NOT valid here,
    # because a bootstrap resample changes which items (and duplicates) enter the pairs.
    # So we rank per replicate, but only over the m pooled rows -> m*(m-1)/2 pairs each.
    out = {(s, e): np.empty(B) for s in spaces for e in encoders}
    enc_names = list(encoders)
    for b in range(B):
        take = items[rng.integers(0, m, m)]                 # resample the pool with replacement
        enc_ranks = {e: standardized_ranks(upper_tri_cos(encoders[e], take))[0].astype(np.float64)
                     for e in enc_names}
        for s in spaces:
            a = standardized_ranks(upper_tri_cos(spaces[s], take))[0].astype(np.float64)
            for e in enc_names:
                out[(s, e)][b] = float(a @ enc_ranks[e])
    return out


# ============================================================== geometry
def geometry(X, sample=3000, seed=0):
    X = np.asarray(X, dtype=np.float64)
    rng = np.random.default_rng(seed)
    idx = rng.choice(X.shape[0], min(sample, X.shape[0]), replace=False)
    Xs = X[idx]
    Xc = Xs - Xs.mean(0)
    s = np.linalg.svd(Xc, compute_uv=False)
    p = s / s.sum()
    p = p[p > 0]
    Xn = Xs / np.linalg.norm(Xs, axis=1, keepdims=True)
    G = Xn @ Xn.T
    iu = np.triu_indices(len(idx), 1)
    return dict(dim=X.shape[1],
                alg_rank=int(np.linalg.matrix_rank(Xc)),
                eff_rank=float(np.exp(-(p * np.log(p)).sum())),
                mean_pair_cos=float(G[iu].mean()),
                mean_norm=float(np.linalg.norm(X, axis=1).mean()))


# ============================================================== cache discovery
def find_file(root, pattern):
    """First file under root whose NAME matches the glob pattern (see find_files)."""
    hits = find_files(root, pattern)
    return hits[0] if hits else None


def find_files(root, pattern):
    """Walk root with os.walk (robust on Google Drive FUSE) and fnmatch on filenames.
    Returns plain string paths to avoid FUSE re-encoding of '+' in filenames."""
    import os as _os
    hits = []
    for dirpath, dirs, files in _os.walk(str(root)):
        for f in files:
            if fnmatch.fnmatch(f, pattern) and (f.endswith(".npz") or f.endswith(".npy")):
                hits.append(_os.path.join(dirpath, f))
    return sorted(hits)


def load_matrix(path, preferred=("img", "txt", "emb", "X", "features")):
    """-> (float64 array, key used, keep array or None)."""
    path = str(path)
    if path.endswith(".npy"):
        return np.asarray(np.load(path), dtype=np.float64), "<npy>", None
    with np.load(path, allow_pickle=False) as z:
        k = pick_array(z, preferred)
        keep = np.asarray(z["keep"]) if "keep" in z.files else None
        return np.asarray(z[k], dtype=np.float64), k, keep


def pick_array(z, preferred=("img", "txt", "emb", "X", "features")):
    """Choose the embedding array inside an npz: the preferred key if present,
    else the only 2-D float array; ambiguity is an error, never a guess."""
    for k in preferred:
        if k in z.files and z[k].ndim == 2:
            return k
    cands = [k for k in z.files if z[k].ndim == 2 and np.issubdtype(z[k].dtype, np.floating)]
    if len(cands) == 1:
        return cands[0]
    raise KeyError(f"ambiguous or missing embedding array; 2-D float keys = {cands}")


# ============================================================== gates
PUBLISHED_ROW_NORM = {  # Results_Summary, 'Cached vectors' table
    "img_small": 54.70, "img_base": 52.91, "img_large": 50.83, "bge": 0.89,
    "gpt2": 207.78, "bert": 9.15, "sbert": 2.43, "convnext": 16.70,
}

PUBLISHED_RHO = {  # E.12 / Table C12m (ConvNeXt row) and the bge-SBERT record
    ("convnext", "img_small"): 0.397, ("convnext", "img_base"): 0.392,
    ("convnext", "img_large"): 0.391, ("convnext", "sbert"): 0.330,
    ("convnext", "bert"): 0.305, ("convnext", "bge"): 0.302,
    ("convnext", "gpt2"): 0.166, ("bge", "sbert"): 0.768,
}


def identity_check(name, X, n_used, tol=0.03):
    """Mean row norm against the published cache table: catches a wrong file or
    wrong key before any statistic is computed. Checked on all rows and on the
    used prefix; either may be the published basis."""
    want = PUBLISHED_ROW_NORM.get(name)
    if want is None:
        return dict(encoder=name, published=None, full=None, prefix=None, ok=None)
    full = float(np.linalg.norm(X, axis=1).mean())
    pre = float(np.linalg.norm(X[:n_used], axis=1).mean())
    ok = min(abs(full - want), abs(pre - want)) / want <= tol
    return dict(encoder=name, published=want, full=round(full, 3), prefix=round(pre, 3), ok=bool(ok))


def reproduction_gate(bank, tol=0.01):
    rows, ok_all = [], True
    for (a, b), want in PUBLISHED_RHO.items():
        if a in bank.r and b in bank.r:
            got = bank.rho(a, b)
            ok = abs(got - want) <= tol
            ok_all &= ok
            rows.append(dict(pair=f"{a}~{b}", published=want, measured=round(got, 4),
                             delta=round(got - want, 4), ok=ok))
        else:
            rows.append(dict(pair=f"{a}~{b}", published=want, measured=None, delta=None, ok=False))
            ok_all = False
    return rows, ok_all


# ============================================================== reality fraction
def reality_fraction(r_frac, r_rand, r_nat, min_bracket=0.02):
    """Where the fractal model sits between its random-init twin (0) and its
    natural-image twin (1), for one reference encoder.

        R = (rho_fractal - rho_random) / (rho_natural - rho_random)

    Undefined when the bracket is narrower than min_bracket: dividing by a
    near-zero gap would produce an enormous, meaningless number -- the same
    arithmetic trap as the width cliff."""
    gap = r_nat - r_rand
    if not np.isfinite(gap) or abs(gap) < min_bracket:
        return np.nan
    return (r_frac - r_rand) / gap


# ============================================================== verdicts
def gap_exceeds(a_reps, b_reps, k=2.0):
    """a > b by more than k combined subsample SDs."""
    d = np.mean(a_reps) - np.mean(b_reps)
    sd = np.sqrt(np.var(a_reps, ddof=1) + np.var(b_reps, ddof=1))
    return bool(d > k * sd), float(d), float(sd)


def verdict(count, total, need):
    if total == 0:
        return "NOT TESTABLE"
    return "CONFIRMED" if count >= need else "FALSIFIED"

In [ ]:
"""G9 I/O: images, pixel baseline, normalization discovery, checkpoint unwrapping.
Torch-free so it can be tested outside Colab."""
import re
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


# ---------------------------------------------------------------- COCO images
def coco_url(image_id, split="train2017"):
    return f"http://images.cocodataset.org/{split}/{int(image_id):012d}.jpg"


def _fetch(args):
    import requests
    image_id, dest, retries = args
    out = Path(dest) / f"{int(image_id):012d}.jpg"
    if out.exists() and out.stat().st_size > 1000:
        return image_id, "cached"
    last = None
    for split in ("train2017", "val2017"):
        for attempt in range(retries):
            try:
                r = requests.get(coco_url(image_id, split), timeout=30)
                if r.status_code == 404:
                    last = "404"
                    break                      # try the other split
                r.raise_for_status()
                tmp = out.with_suffix(".part")
                tmp.write_bytes(r.content)
                tmp.rename(out)
                return image_id, split
            except Exception as e:           # noqa: BLE001 - reported, not swallowed
                last = f"{type(e).__name__}: {e}"
                time.sleep(1.5 * (attempt + 1))
    return image_id, f"FAILED ({last})"


def download_images(ids, dest, workers=32, retries=3, log_every=1000):
    dest = Path(dest)
    dest.mkdir(parents=True, exist_ok=True)
    status = {}
    with ThreadPoolExecutor(workers) as ex:
        for i, (iid, st) in enumerate(ex.map(_fetch, [(i, dest, retries) for i in ids]), 1):
            status[iid] = st
            if i % log_every == 0:
                print(f"  {i}/{len(ids)}")
    failed = {k: v for k, v in status.items() if v.startswith("FAILED")}
    splits = {}
    for v in status.values():
        splits[v] = splits.get(v, 0) + 1
    return status, failed, splits


def load_and_crop(path, resize=256, crop=224):
    """Shorter side -> resize (bilinear, PIL antialiases when downsampling),
    then centre crop. The standard 0.875 crop every model here expects."""
    from PIL import Image
    with Image.open(path) as im:
        im = im.convert("RGB")
        w, h = im.size
        s = resize / min(w, h)
        im = im.resize((max(crop, round(w * s)), max(crop, round(h * s))), Image.BILINEAR)
        w, h = im.size
        l, t = (w - crop) // 2, (h - crop) // 2
        return np.asarray(im.crop((l, t, l + crop, t + crop)), dtype=np.uint8)


def build_crop_cache(ids, img_dir, out_path, workers=8, crop=224):
    """uint8 memmap [N, crop, crop, 3] in the EXACT order of ids -- the order is
    the alignment, so nothing is ever skipped or reordered."""
    out_path = Path(out_path)
    N = len(ids)
    arr = np.lib.format.open_memmap(out_path, mode="w+", dtype=np.uint8, shape=(N, crop, crop, 3))
    paths = [Path(img_dir) / f"{int(i):012d}.jpg" for i in ids]
    missing = [p for p in paths if not p.exists()]
    if missing:
        raise FileNotFoundError(f"{len(missing)} images missing, e.g. {missing[:3]} -- refusing to "
                                "build a cache with holes: positional alignment would break")

    def work(k):
        arr[k] = load_and_crop(paths[k], crop=crop)
        return k
    with ThreadPoolExecutor(workers) as ex:
        for j, _ in enumerate(ex.map(work, range(N)), 1):
            if j % 2000 == 0:
                print(f"  cropped {j}/{N}")
    arr.flush()
    return out_path


def pixel_features(crops, size=32, batch=512):
    """Block-mean downsample to size x size RGB, flatten, then centre each
    feature across the dataset so cosine compares layouts, not brightness."""
    N, H, W, C = crops.shape
    f = H // size
    out = np.empty((N, size * size * C), dtype=np.float32)
    for i in range(0, N, batch):
        x = np.asarray(crops[i:i + batch], dtype=np.float32) / 255.0
        x = x[:, :size * f, :size * f].reshape(len(x), size, f, size, f, C).mean((2, 4))
        out[i:i + batch] = x.reshape(len(x), -1)
    return out - out.mean(0, keepdims=True)


# ---------------------------------------------------------------- normalization discovery
_NUM = r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?"
_PAT = [
    re.compile(r"Normalize\(\s*(?:mean\s*=\s*)?[\[\(]([^\]\)]+)[\]\)]\s*,\s*(?:std\s*=\s*)?[\[\(]([^\]\)]+)[\]\)]"),
    re.compile(r"mean\s*[:=]\s*[\[\(]([^\]\)]+)[\]\)][\s\S]{0,200}?std\s*[:=]\s*[\[\(]([^\]\)]+)[\]\)]"),
]


def _nums(s):
    v = [float(x) for x in re.findall(_NUM, s)]
    return tuple(v) if len(v) == 3 else None


def find_normalizations(repo_dir, prefer=("pretrain", "fractal")):
    """Every (file, mean, std) triple the repo declares, preferred files first.
    The caller prints them all; nothing is chosen silently."""
    hits = []
    for p in sorted(Path(repo_dir).rglob("*")):
        if p.suffix not in (".py", ".yaml", ".yml") or not p.is_file():
            continue
        txt = p.read_text(errors="ignore")
        for pat in _PAT:
            for m in pat.finditer(txt):
                mean, std = _nums(m.group(1)), _nums(m.group(2))
                if mean and std:
                    score = sum(k in str(p).lower() for k in prefer)
                    hits.append(dict(file=str(p.relative_to(repo_dir)), mean=mean, std=std, score=score))
    hits.sort(key=lambda h: -h["score"])
    seen, uniq = set(), []
    for h in hits:
        key = (h["file"], h["mean"], h["std"])
        if key not in seen:
            seen.add(key)
            uniq.append(h)
    return uniq


# ---------------------------------------------------------------- checkpoints
def unwrap_state_dict(obj, is_tensor):
    """Peel the usual containers and DataParallel prefixes. Returns
    (state_dict, notes) so the notebook can print exactly what was done."""
    notes = []
    for k in ("state_dict", "model", "model_state_dict", "net", "module"):
        if isinstance(obj, dict) and k in obj and isinstance(obj[k], dict):
            obj = obj[k]
            notes.append(f"unwrapped '{k}'")
    sd = {}
    stripped = set()
    for k, v in obj.items():
        if not is_tensor(v):
            continue
        for p in ("module.", "backbone."):
            if k.startswith(p):
                k = k[len(p):]
                stripped.add(p)
        sd[k] = v
    if stripped:
        notes.append(f"stripped prefixes {sorted(stripped)}")
    return sd, notes

In [ ]:
"""PRH evidence table: every measure in the project, ordered by the proof step it
serves, plus the G9 fractal rows. Existing rows are transcribed from
Final_Project_Report.pdf and Results_Summary.pdf (section refs in 'source');
the report is the source of truth and wins any disagreement with this file.
"""
import pandas as pd

STEPS = [
    ("S0", "Instrument validity", "Before any PRH claim: are the rows aligned, the fits powered, the nulls calibrated?"),
    ("S1", "'Well-trained' - learned, not architectural", "Similarity must come from training, not from the architecture alone (random-weights control)."),
    ("S1b", "'Well-trained' ON REALITY - is it the data?", "NEW (G9). PRH says convergence tracks the statistics of the world. A model trained only on fractals has training without reality."),
    ("S2", "'Independent' - no shared ancestry or recipe", "Similarity must survive different lineage, architecture and supervision, or it is inherited, not discovered."),
    ("S3", "'Up to a linear transformation'", "The map class is the crux: linear must suffice; rotation-only is the stronger sub-claim."),
    ("S4", "Across modalities - the 'Platonic' claim proper", "An image-only and a text-only model must share structure though they never saw each other's data."),
    ("S5", "'As models grow stronger' - scaling", "PRH predicts alignment rises with capacity."),
    ("S6", "Mechanism - what drives convergence", "Which geometric property makes spaces alignable, and does it predict?"),
    ("S7", "Self-identifying correspondence", "Can the correspondence be found from geometry alone, with no paired examples?"),
    ("S8", "One shared space, used", "Demonstration by use: a component trained in one space works in another."),
    ("S9", "Boundaries and falsifications", "What the evidence does not support, and explanations that were withdrawn."),
]
STEP_NAME = {s: n for s, n, _ in STEPS}
STEP_ORDER = {s: i for i, (s, _, _) in enumerate(STEPS)}

COLS = ["step", "id", "measure", "experiment", "result", "control_or_chance",
        "what_it_proves", "new", "source"]

R = lambda *a: dict(zip(COLS, a))

EXISTING = [
    # ---------------------------------------------------------------- S0
    R("S0", "0.1", "Row alignment by shuffle test",
      "All caches, 9,533 COCO train2017 items; ridge or rank correlation intact vs row-shuffled. Two id namespaces (request position vs COCO id) never joined by id.",
      "Honest fits beat shuffled by 0.545-0.863 (threshold 0.2); scale-extension encoders 0.26-0.85",
      "Shuffled = 0.00",
      "Row i is the same item in every cache - a precondition, not PRH evidence",
      "Project method", "E.15; H1"),
    R("S0", "0.2", "Artifact certification (H1)",
      "47 automated checks over every cache: shapes, norms, alignment, rows/dim",
      "No hard failures; 2 flags, both the documented GPT-2 collapse",
      "-", "Caches are well-formed and aligned", "Project method", "D.5; H1"),
    R("S0", "0.3", "Statistical power: rows per input dimension",
      "Every fit checked against a pre-registered floor of 5 rows/dim. Exp A run 1 (1,646 samples) vs run 2 (10,000), identical pipeline",
      "All fits 5.9-9.8 rows/dim; Exp A run 1 FAILED on sample starvation, run 2 PASSED",
      "Floor = 5", "Failures are diagnosable as power, not as absence of convergence",
      "Methodological contribution (report s.5)", "s.3.2-3.3; H1"),
    R("S0", "0.4", "Null calibration by permutation (G5)",
      "Permutation nulls for k-NN overlap and CKA across 21 encoder pairs",
      "k-NN null = k/N exactly; CKA null = +0.094 and rises with width. Correcting it widens objective-over-lineage gap +0.024 -> +0.181",
      "Permutation null", "Uncorrected CKA manufactures agreement that favoured the lineage pair",
      "Known bias; its effect on this project's conclusion is project-specific", "G5; C.13.9"),

    # ---------------------------------------------------------------- S1
    R("S1", "1.1", "Linear CKA, trained vs random weights (Exp A)",
      "GPT-2 vs Pythia-160M (deliberately hostile pair), WikiText-103 10k passages, layerwise CKA",
      "Diagonal mean 0.424 (pass >0.35); trained/random 6.3x (pass >=3x)",
      "Random-weights control 0.067 (pass <0.15)",
      "Similarity is produced by training, not by architecture",
      "Replication (Kornblith 2019) on a hostile pair", "s.3.4; C.7"),
    R("S1", "1.2", "Stitching R2 and depth signature (Exp A)",
      "Same pair; one linear map per layer, held-out R2",
      "Peak L11 0.797 (pass >0.70); trained curve rises 0.59 -> 0.80 with depth",
      "Random 0.173 at L11 and DECAYS 0.38 -> 0.09 - opposite slope, 8x gap",
      "Convergence builds with depth - learned, not inherited",
      "Replication (Bansal 2021)", "s.3.4"),

    # ---------------------------------------------------------------- S2
    R("S2", "2.1", "Cross-lineage transfer (G4)",
      "SigLIP 2 (independent lineage, no DINOv2 ancestry) given one map into the frozen 4-space hub; frozen caption head",
      "94.2% of a natively-fitted head - inside the 93.8-96.5% within-family band",
      "Random-map control at chance",
      "Shared space is not a family trait",
      "Extension (cross-lineage, not simultaneously cross-objective; still a ViT)", "C.13; G4; G0-exact"),
    R("S2", "2.2", "Cross-architecture transfer (E.2)",
      "ConvNeXt-base (convnet, ImageNet-22k supervised) held out of the hub, one entry map",
      "96.7% of native - ABOVE the band and above SigLIP 2 in the same run; 99.3% in the 7-space hub (G10)",
      "Control 0.001 = chance",
      "Architecture does not bound the claim (objective still confounded)",
      "New in this project", "E.2; G10"),
    R("S2", "2.3", "Supervision regime (E.10)",
      "Label-free (DINOv2, GPT-2, BERT), pair-supervised (bge, SigLIP) and 22k-class supervised (ConvNeXt) encoders in one hub",
      "All carry a head at 93-97% of native",
      "-", "Convergence does not require a shared supervision signal - keeps the thesis non-circular",
      "New in this project", "E.10"),
    R("S2", "2.4", "What explains pairwise agreement (G5, exploratory)",
      "21 pairs grouped by shared objective / lineage / modality",
      "Objective +53x > lineage +44x > modality +29x; top pair bge-SBERT shares objective, not lineage",
      "Permutation-calibrated",
      "Supervision modulates HOW MUCH encoders agree, not WHETHER",
      "New in this project (exploratory)", "G5; E.10"),

    # ---------------------------------------------------------------- S3
    R("S3", "3.1", "MLP vs ridge, controlled (F1/F2)",
      "Distillation corpus 38k rows (74.2 rows/dim), MLP vs closed-form ridge, two independent runs",
      "Held-out cosine 0.9183 = 0.9183, gain +0.0000, both runs",
      "Same data, same split",
      "Convergence is linear - capacity beyond a matrix adds nothing",
      "New in this project (controlled test)", "C.9; F1/F2"),
    R("S3", "3.2", "Adapter: linear vs nonlinear (Exp B)",
      "MobileCLIP-S1 -> SigLIP 2, 4,000 COCO val2017, 3,000/1,000 split",
      "Ridge cosine 0.900; MLP 0.903 (+0.003)",
      "-", "Deployed relationship is linear", "Consistent with stitching literature", "s.4; B2"),
    R("S3", "3.3", "Related, not rigid: rotation-only vs linear",
      "Orthogonal Procrustes vs ridge: Exp B pair; 21 hub pairs (G6); raw-space similarity transform (K1)",
      "Procrustes R2 0.038 vs ridge 0.592 (Exp B), 69x singular-value spread; 0.194 vs 0.667 over 21 pairs; similarity transform 0.281 vs 0.667",
      "-", "The strong (rotation-only) sub-claim FAILS: spaces are linearly related but anisotropic",
      "New in this project", "s.4; G6; K1"),

    # ---------------------------------------------------------------- S4
    R("S4", "4.1", "Image-only -> text-only retrieval (C.8)",
      "DINOv2-base (never saw text) -> bge-m3 (never saw an image); 8,000 COCO train2017 pairs, 9.1 rows/dim, ridge",
      "R2 0.511; R@1 0.358 = 40.9% of a matched SigLIP ceiling (0.875); pre-registered verdict: measurable but weak",
      "358x chance; raw cross-space floor 0.001; shuffle gap 0.641",
      "Cross-modal structure exists without joint training",
      "Replication of PRH's cross-modal claim at small scale", "C.8"),
    R("S4", "4.2", "Raw cross-modal shape agreement",
      "Spearman of pairwise cosine distances, 8 encoders, all 9,533 items, no map",
      "Image-text mean rho 0.290 (8 encoders); top pair bge-SBERT 0.768; contrastive text pairs reach 0.76-0.86 in the scale extension",
      "rho ~ 0 unrelated",
      "Cross-modal agreement is real but modest; same-objective text agrees far more",
      "Replication (PRH-style measurement)", "C.13.9; E.12; E.16"),
    R("S4", "4.3", "Shared subspace size (G6)",
      "CCA across pairs; intrinsic-dimension estimates",
      "Cross-modal pairs share as few as 6 of 64 CCA directions; intrinsic dimension 11.2-19.9 inside 768-2048-d spaces (<3%)",
      "-", "What is shared is a small, low-dimensional structure", "New in this project", "G6"),
    R("S4", "4.4", "Reach into the shared hub (E.6)",
      "Each encoder's held-out R2 for its own map into the 7-space hub",
      "Image encoders 0.46-0.57; all four text spaces 0.02-0.05, uniformly",
      "-", "A whitened hub is largely an image-side object; text touches few of its directions",
      "New in this project", "E.6"),

    # ---------------------------------------------------------------- S5
    R("S5", "5.1", "Capacity sweep (C.10)",
      "DINOv2 small/base/large -> bge, pooling and rows/dim (11.1) held constant",
      "40.9 / 53.5 / 58.3% of ceiling; +15.3 points per decade of parameters; fit R2 0.949; six monotone measures",
      "Matched ceiling",
      "Alignment rises with capacity in this range",
      "Replication of PRH scaling at small scale", "C.10"),
    R("S5", "5.2", "Pre-registered scale extension (E.16)",
      "DINOv2-giant (cls+patch) and Qwen-4B added to matched ladders; thresholds fixed before running",
      "Giant rho 0.339 < 0.418 FALSIFIED; Qwen-4B rho 0.307 < 0.320 FALSIFIED; both ladders rise then fall",
      "Pooling artifact ruled out (cls-only 0.238 discarded)",
      "Naive 'more capacity -> more alignment' is NOT supported at this scale",
      "New in this project (pre-registered negative)", "E.16"),
    R("S5", "5.3", "Raw agreement vs transfer under scaling (E.16 P4)",
      "Same ladders, both quantities",
      "delta-rho -0.079 while delta-transfer +7.1 points - opposite directions",
      "-", "Scaling moves the two quantities apart", "New in this project", "E.16"),

    # ---------------------------------------------------------------- S6
    R("S6", "6.1", "Isotropy rescue (C.11)",
      "GPT-2 caption space (effective rank 6.5) whitened, label-free, fitted on training rows only",
      "R@1 0.130 -> 0.478 = 102.6% of a 1.2B-pair contrastive encoder (101.4% at a second scale); eff. rank 6.5 -> 490.8; iso-retrieval corr +0.94",
      "Same protocol, unwhitened",
      "Isotropy, not similarity training, is what a space needs to be alignable",
      "New in this project", "C.11; E1.2"),
    R("S6", "6.2", "Encoder ladder as a prediction (C.14)",
      "GPT-2 / BERT / SBERT as text side, same image encoder, pre-registered",
      "Eff. rank 6.5 / 65.0 / 93.4 -> R@1 0.160 / 0.362 / 0.510; rank corr +1.00 (P1 confirmed); P2 partial",
      "-", "The mechanism predicts before measuring; isotropy necessary, not sufficient",
      "New in this project", "C.14"),
    R("S6", "6.3", "Meta-finding: read vs find",
      "C.11 (known correspondence) vs C.12 (unknown correspondence)",
      "Whitening helps reading (+102.6%) and destroys finding (39.7% -> 0.1%)",
      "-", "One geometric property, opposite prescriptions - two instances (corrected down from three)",
      "New in this project", "C.11; C.12; E.8"),

    # ---------------------------------------------------------------- S7
    R("S7", "7.1", "Pair-free recovery by Gromov-Wasserstein (C.12)",
      "Two contrastive image-text encoders, zero paired examples",
      "39.7% exact matches; map fitted from that assignment R@1 0.252 (40% of ceiling), AUC 0.906",
      "595x chance (0.067%)",
      "The correspondence is self-identifying from geometry alone - bounded: weakest-independence pair",
      "Replication in spirit (vec2vec 2025) by a different method", "C.12"),

    # ---------------------------------------------------------------- S8
    R("S8", "8.1", "Zero-shot hub transfer (C.13)",
      "5 spaces over 9,533 images, one linear map each into a 512-d whitened hub; caption head trained on img_small only",
      "img_base 95.9%, img_large 92.9% of native - absolute R@1 RISES on unseen encoders",
      "Random-map control 0.000 (chance 0.001)",
      "The shared space exists by use, not only by correlation",
      "New in this project", "C.13"),
    R("S8", "8.2", "Composition and compatibility (G1, G3)",
      "7-space hub; all 42 ordered pairs vs bespoke hubs",
      "95.6% retention over 36/41 pairs (exceptions all write INTO GPT-2); 42/42 pairs shared, mean 103.6%; GPT-2 source 95.1% vs target 24.8%",
      "Control 0.004 vs chance 0.001",
      "One hub serves all members; collapsed spaces are readable out of, not writable into",
      "New in this project", "C.13; G1; G3"),
    R("S8", "8.3", "Deployed adapter (Exp B)",
      "MobileCLIP-S1 -> SigLIP 2 ridge matrix, shared Qdrant index, Core ML fp16",
      "R@1/5/10 = 88.4 / 95.3 / 97.2% of ceiling; verification AUC 0.999; hubness skew 4.28 -> 1.61; 0.79 MB",
      "Raw cross-space baseline = exact chance",
      "PRH is load-bearing in a working two-tier system",
      "Uncommon application (report s.5)", "s.4; D.2; D.3"),

    # ---------------------------------------------------------------- S9
    R("S9", "9.1", "Local, not global (G5)",
      "CKNNA (the literature's metric) vs the project's own k-NN measure, 21 pairs",
      "21/21 pairs higher locally than at the global limit; the two metrics agree at Spearman +0.81",
      "k-NN null = k/N",
      "Agreement is concentrated in local neighbourhoods",
      "Replication of CKNNA + project finding", "G5; C.13.9"),
    R("S9", "9.2", "Raw geometry does not predict transfer (E.12)",
      "ConvNeXt's raw rho vs its hub transfer; second instance of E.6",
      "Lowest raw rho of any image encoder (0.391-0.397 to DINOv2, 0.166-0.330 to text) yet best transfer 96.7%",
      "-", "Raw shape agreement bounds what is VISIBLE without fitting, not what is RECOVERABLE",
      "New in this project", "E.12; E.6"),
    R("S9", "9.3", "Width cliff explained (E.8)",
      "Hub-width sweep; retained-rank sweep in steps of 8",
      "Step at the source's ambient dimension, 6/6 curves; sigma 0.0054 vs 10.79; holds to k=760 then 0.009; repaired by truncation (0.878) or head alpha 1e3 (0.917)",
      "Three off-dimension widths: nothing",
      "A numerical artifact of least squares, NOT a property of shared representations",
      "New in this project", "E.3; E.8; G11-G16"),
    R("S9", "9.4", "Falsification ledger",
      "Every proposed explanation tested and recorded",
      "Nine findings; eleven explanations falsified, two of them the project's own",
      "-", "Claims that survived did so against recorded alternatives",
      "Project practice", "s.7.4; Results_Summary"),
    R("S9", "9.5", "Scope boundary",
      "Roster width vs PRH's evidence base",
      "Encoders here are 768-2,048-d; PRH's evidence extends to 70B-parameter LLMs",
      "-", "Results bound the claim at this scale; they do not test PRH's large-model regime",
      "Boundary", "E.16; viva brief"),
]

# ---------------------------------------------------------------- S1b (G9) templates
G9_TEMPLATE = [
    R("S1b", "F.1", "Shape agreement: FractalDB ResNet-50 vs each of the 8 encoders",
      "ResNet-50 trained ONLY on FractalDB-1k (1M rendered fractals, 1k formula-defined classes, zero natural images); 2048-d global-average-pooled features of the same 9,533 COCO images; Spearman of pairwise cosine, all items, no map",
      "{F1}", "{F1c}",
      "{F1p}", "Candidate new - no prior test of FDSL models against a cross-modal PRH roster found in searches", "G9 (this notebook)"),
    R("S1b", "F.2", "Shape agreement: FractalDB DeiT vs each of the 8 encoders",
      "DeiT (ViT) trained ONLY on FractalDB-1k; cls+patch pooling to match the DINOv2 caches; same items and statistic",
      "{F2}", "{F2c}",
      "{F2p}", "Candidate new (as F.1)", "G9"),
    R("S1b", "F.3", "Brackets: random-init and ImageNet twins, pixel floor",
      "Same two architectures with random weights (lower bracket) and trained on ImageNet-1k (upper bracket); 32x32 raw pixels as the floor",
      "{F3}", "Brackets ARE the controls",
      "Places the fractal result on a scale from 'architecture only' to 'architecture + reality'", "Project design", "G9"),
    R("S1b", "F.4", "Reality fraction R, image vs text encoders",
      "R = (rho_fractal - rho_random) / (rho_natural - rho_random) per encoder; undefined if the bracket < 0.02",
      "{F4}", "{F4c}",
      "{F4p}", "Candidate new metric", "G9"),
    R("S1b", "F.5", "Sensitivity: pooling and normalization",
      "DeiT cls-only vs cls+patch; repo normalization vs ImageNet normalization",
      "{F5}", "-",
      "Checks the result is not a preprocessing artifact (the giant cls vs cls+patch lesson, E.16)", "Project practice", "G9"),
    R("S1b", "F.6", "Whitening rescue of the collapsed fractal CNN (C.11 move)",
      "frac_cnn / rand_cnn are collapsed (mean pair-cos 0.85 / 1.00); PCA-whiten each bracket space and re-measure the same Spearman rho. Row correspondence is known, so this is the READING regime (C.11), not the FINDING regime (C.12). Secondary diagnostic; the pre-registered raw verdicts stand.",
      "{F6}", "{F6c}",
      "{F6p}", "Third instance of the E.6/E.12 lesson (after GPT-2 and ConvNeXt): raw agreement on a degenerate space measures the post-collapse residual, not what is recoverable", "G9"),
]

G9_PENDING = {
    "F1": "PENDING - run G9. Pre-registered P1: rho_fractal < rho_ImageNet-twin on >=7 of 8 encoders, gap > 2 combined SDs",
    "F1c": "Random-init ResNet-50 twin; row-shuffle null",
    "F1p": "If P1 holds: convergence needs natural-image statistics, not just training - the PRH mechanism. If it fails: training on ANY structured data suffices, which weakens the 'reality' reading",
    "F2": "PENDING - same P1 for the ViT",
    "F2c": "Random-init DeiT twin; row-shuffle null",
    "F2p": "Repeats F.1 on a second architecture, so the verdict cannot be a CNN idiosyncrasy",
    "F3": "PENDING. Pre-registered P2: rho_fractal > rho_random on >=3 of 4 image encoders; P5: rho_fractal > rho_pixels on >=3 of 4 image encoders",
    "F4c": "Subsample interval (30 x 2,000 items); R undefined where the bracket < 0.02",
    "F4": "PENDING. Pre-registered P3: mean R over healthy TEXT encoders < mean R over IMAGE encoders (GPT-2 stratified out)",
    "F4p": "If P3 holds: fractals teach generic visual structure, but the structure shared with language needs the world - the sharpest PRH-specific test here",
    "F5": "PENDING. Flag if |delta-rho| > 0.02",
    "F6": "PENDING - run G9 cells 17-18. Raw P5-cnn FALSIFIED (frac_cnn at the pixel floor) may flip to CONFIRMED once collapse is whitened away, if the masked signal is real",
    "F6c": "Random-init twin and 32x32 pixels as floors, both whitened the same way; row-shuffle null",
    "F6p": "If whitening flips P5-cnn to CONFIRMED: the raw CNN 'failure' was collapse, and the fractal CNN does carry a small real slice of shared structure (size = R_white, typically << natural images). If it stays FALSIFIED: the fractal CNN genuinely shares nothing at this pooling",
}


def fill_g9(values):
    rows = []
    for r in G9_TEMPLATE:
        rr = dict(r)
        for c in COLS:
            v = rr[c]
            if isinstance(v, str) and v.startswith("{") and v.endswith("}"):
                rr[c] = values.get(v[1:-1], G9_PENDING[v[1:-1]])
        rows.append(rr)
    return rows


def build_table(g9_values=None):
    df = pd.DataFrame(EXISTING + fill_g9(g9_values or {}), columns=COLS)
    df["step_name"] = df["step"].map(STEP_NAME)
    df["_o"] = df["step"].map(STEP_ORDER)
    df["_i"] = df["id"].str.replace("F.", "0.", regex=False).astype(float)
    df = df.sort_values(["_o", "_i"]).drop(columns=["_o", "_i"])
    return df[["step", "step_name"] + COLS[1:]].reset_index(drop=True)


def write_xlsx(df, path, title="PRH evidence table"):
    from openpyxl import Workbook
    from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
    from openpyxl.utils import get_column_letter

    heads = ["PRH step", "Step", "ID", "Measure", "Experiment (models, data, protocol)", "Result",
             "Control / chance", "What it proves for PRH", "New?", "Source"]
    widths = [7, 24, 6, 28, 46, 46, 26, 40, 24, 16]
    wb = Workbook()
    ws = wb.active
    ws.title = "PRH evidence"
    thin = Side(style="thin", color="BFBFBF")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    ws["A1"] = title
    ws["A1"].font = Font(name="Arial", bold=True, size=14)
    ws["A2"] = ("Ordered by the PRH proof step each measure serves. Existing rows transcribed from "
                "Final_Project_Report.pdf / Results_Summary.pdf (the report is the source of truth). "
                "S1b rows are the G9 fractal experiment: PENDING until the notebook runs, then filled from its output.")
    ws["A2"].font = Font(name="Arial", italic=True, size=9)
    ws.merge_cells("A2:J2")
    ws["A2"].alignment = Alignment(wrap_text=True, vertical="top")
    ws.row_dimensions[2].height = 30
    for j, (h, w) in enumerate(zip(heads, widths), 1):
        c = ws.cell(row=4, column=j, value=h)
        c.font = Font(name="Arial", bold=True, color="FFFFFF", size=10)
        c.fill = PatternFill("solid", fgColor="1F3864")
        c.alignment = Alignment(wrap_text=True, vertical="center")
        c.border = border
        ws.column_dimensions[get_column_letter(j)].width = w
    band = {s: ("F2F2F2" if i % 2 else "FFFFFF") for i, (s, _, _) in enumerate(STEPS)}
    for i, row in enumerate(df.itertuples(index=False), 5):
        new_row = row.step == "S1b"
        for j, v in enumerate(row, 1):
            c = ws.cell(row=i, column=j, value=v)
            c.font = Font(name="Arial", size=9, bold=(j in (1, 3, 4)))
            c.alignment = Alignment(wrap_text=True, vertical="top")
            c.border = border
            c.fill = PatternFill("solid", fgColor="FFF2CC" if new_row else band[row.step])
    ws.freeze_panes = "E5"
    ws.auto_filter.ref = f"A4:J{4 + len(df)}"

    ws2 = wb.create_sheet("PRH steps")
    for j, h in enumerate(["Step", "Name", "What must be shown"], 1):
        c = ws2.cell(row=1, column=j, value=h)
        c.font = Font(name="Arial", bold=True, color="FFFFFF")
        c.fill = PatternFill("solid", fgColor="1F3864")
    for i, (s, n, d) in enumerate(STEPS, 2):
        for j, v in enumerate((s, n, d), 1):
            c = ws2.cell(row=i, column=j, value=v)
            c.font = Font(name="Arial", size=10, bold=(j == 1))
            c.alignment = Alignment(wrap_text=True, vertical="top")
    for col, w in zip("ABC", (8, 44, 90)):
        ws2.column_dimensions[col].width = w
    wb.save(path)

## 4 — Self-test — **stop here if anything says FAIL**

The engine must equal `scipy.stats.spearmanr` (including the tie path and a collapsed space),
be rotation-invariant, give ≈0 under shuffle, and the reality fraction must refuse a narrow
bracket rather than divide by it.

In [ ]:
from scipy.stats import ortho_group
_f = []
def _chk(n, ok, d=""):
    print(f"{'PASS' if ok else 'FAIL'}  {n:<52} {d}")
    if not ok: _f.append(n)
_r = np.random.default_rng(0); _n = 400
_S = _r.normal(size=(_n, 24))
_A = _S @ _r.normal(size=(24, 80)) + 0.5 * _r.normal(size=(_n, 80))
_B = _S @ _r.normal(size=(24, 60)) + 0.9 * _r.normal(size=(_n, 60))
_Cc = np.outer(np.ones(_n), _r.normal(size=40)) * 50 + 0.05 * (_S @ _r.normal(size=(24, 40)))
for nm, (X, Y) in {"shared": (_A, _B), "collapsed": (_Cc, _A)}.items():
    got = rho_direct(X, Y); want = spearmanr(upper_tri_cos(X), upper_tri_cos(Y)).statistic
    _chk(f"engine == scipy [{nm}]", abs(got - want) < 1e-5, f"{got:.6f} vs {want:.6f}")
_v = np.round(_r.normal(size=3000), 1); _w = _v + _r.normal(size=3000)
_a, _ = standardized_ranks(_v); _b, _ = standardized_ranks(_w)
_chk("tie path == scipy average ranks", abs(float(_a @ _b) - spearmanr(_v, _w).statistic) < 1e-5)
_Q = ortho_group.rvs(80, random_state=1)
_chk("rotation invariance", abs(rho_direct(_A, _A @ _Q) - 1) < 1e-9)
_null = subsample_rhos(_A, _B, m=200, B=10, shuffle=True)
_chk("shuffle null ~ 0", abs(_null.mean()) < 0.02, f"{_null.mean():+.4f}")
_chk("R narrow bracket -> NaN", np.isnan(reality_fraction(0.21, 0.20, 0.205)))
_chk("R midpoint = 0.5", abs(reality_fraction(0.35, 0.2, 0.5) - 0.5) < 1e-12)
_bk = RankBank(); _bk.add("A", _A); _bk.add("B", _B)
_chk("RankBank == direct", abs(_bk.rho("A", "B") - rho_direct(_A, _B)) < 1e-6)
print("\nALL SELF-TESTS PASSED" if not _f else f"\nFAILURES: {_f}  -- do not continue")
assert not _f

## 5 — Synthetic data (dry run only)

Writes fake caches with the real file names, key names and published row norms, and a latent
structure in which natural > fractal > random > pixels. Skipped when `SYNTHETIC = False`.

In [ ]:
def make_synthetic(root, n, seed=0):
    rng = np.random.default_rng(seed)
    root.mkdir(parents=True, exist_ok=True)
    world = rng.normal(size=(n + 400, 24))           # what the world contains
    low = rng.normal(size=(n + 400, 16))             # low-level image statistics
    def enc(d, w_world, w_low, noise, norm, rows):
        X = (w_world * world[:rows] @ rng.normal(size=(24, d)) +
             w_low * low[:rows] @ rng.normal(size=(16, d)) + noise * rng.normal(size=(rows, d)))
        return X / np.linalg.norm(X, axis=1, keepdims=True).mean() * norm
    ids = np.sort(rng.choice(np.arange(9, 60000), n, replace=False))
    np.savez(root / "e1_img_ckpt_dinov2-small_cls+patch.npz", img=enc(768, 1, .6, .8, 54.70, n), keep=np.arange(n))
    np.savez(root / "e1_img_ckpt_dinov2-base_cls+patch.npz", img=enc(1536, 1, .6, .7, 52.91, n + 100), keep=np.arange(n + 100))
    np.savez(root / "e1_img_ckpt_dinov2-large_cls+patch.npz", img=enc(2048, 1, .6, .7, 50.83, n + 200), keep=np.arange(n + 200))
    np.savez(root / "e1_img_ckpt_convnext-base-224-22k.npz", img=enc(1024, 1, .9, .9, 16.70, n), keep=ids)
    np.savez(root / "crossmodal_pairs.npz", img=enc(1536, 1, .6, .7, 1, n + 50), txt=enc(1024, .7, 0, 1.0, 0.89, n + 50))
    g = enc(768, .5, 0, 1.0, 1, n + 50); g = g + 40 * rng.normal(size=768)       # collapsed
    np.savez(root / "crossmodal_pairs_gpt2.npz", txt=g / np.linalg.norm(g, axis=1, keepdims=True).mean() * 207.78)
    np.savez(root / "e13_txt_bert.npz", txt=enc(768, .6, 0, 1.0, 9.15, n + 50))
    np.savez(root / "e13_txt_sbert.npz", txt=enc(768, .7, 0, 1.0, 2.43, n + 50))
    new = dict(nat_cnn=enc(2048, 1, .8, .8, 1, n), nat_vit=enc(1536, 1, .7, .8, 1, n),
               frac_cnn=enc(2048, .35, 1, .9, 1, n), frac_vit=enc(1536, .3, 1, .9, 1, n),
               rand_cnn=enc(2048, .05, 1, 1, 1, n), rand_vit=enc(1536, .03, .8, 1.2, 1, n),
               pixels=enc(3072, 0, .7, 1.4, 1, n))
    new["frac_vit_cls"] = new["frac_vit"][:, :768] + 0.05 * rng.normal(size=(n, 768))
    return new

SYN_NEW = make_synthetic(DATA_DIR, N_ITEMS) if SYNTHETIC else None
print("synthetic caches written" if SYNTHETIC else "real run - skipped")

## 6 — Load the eight encoders, identity-check, and reproduce the published ρ

File patterns come from Results_Summary's cache table. Two gates, both before any new number:

1. **Identity** — mean row norm vs the published table (3% tolerance) catches a wrong file or key.
2. **Reproduction** — the ConvNeXt row of Table C12m and bge–SBERT 0.768, ±0.01. If this fails,
   every verdict below is stamped **UNVERIFIED**.

In [ ]:
ENC = {  # name: (filename glob, preferred keys, modality)
    "img_small": ("e1_img_ckpt_dinov2-small_cls+patch*", ("img",), "image"),
    "img_base":  ("e1_img_ckpt_dinov2-base_cls+patch*",  ("img",), "image"),
    "img_large": ("e1_img_ckpt_dinov2-large_cls+patch*", ("img",), "image"),
    "convnext":  ("e1_img_ckpt_convnext-base-224-22k*",  ("img",), "image"),
    "bge":       ("crossmodal_pairs.npz",                ("txt",), "text"),
    "gpt2":      ("crossmodal_pairs_gpt2.npz",           ("txt",), "text"),
    "bert":      ("e13_txt_bert*",                       ("txt", "emb"), "text"),
    "sbert":     ("e13_txt_sbert*",                      ("txt", "emb"), "text"),
}
IMG_ENC  = [k for k, v in ENC.items() if v[2] == "image"]
TXT_ENC  = [k for k, v in ENC.items() if v[2] == "text"]
DEGENERATE = {"gpt2"}          # documented collapse (C.11, E.10); confirmed by geometry below

E, KEEP, rows = {}, {}, []
for name, (pat, pref, mod) in ENC.items():
    hits = find_files(DATA_DIR, pat)
    if not hits:
        raise FileNotFoundError(f"{name}: no file matching '{pat}' under {DATA_DIR}")
    if len(hits) > 1:
        print(f"  note: {name} pattern matched {len(hits)} files, using {Path(hits[0]).name}: {[Path(h).name for h in hits]}")
    X, key, keep = load_matrix(hits[0], pref)
    if X.shape[0] < N_ITEMS:
        raise ValueError(f"{name}: {X.shape[0]} rows < N_ITEMS={N_ITEMS}")
    ident = identity_check(name, X, N_ITEMS)
    E[name], KEEP[name] = X[:N_ITEMS], (None if keep is None else keep[:N_ITEMS])
    rows.append(dict(encoder=name, file=Path(hits[0]).name, key=key, rows_in_file=X.shape[0], dim=X.shape[1],
                     norm_published=ident["published"], norm_file=ident["full"],
                     norm_prefix=ident["prefix"], identity_ok=ident["ok"]))
ident_df = pd.DataFrame(rows)
print(ident_df.to_string(index=False))
if not ident_df.identity_ok.all():
    print("\n!! IDENTITY CHECK FAILED for", list(ident_df.loc[~ident_df.identity_ok, "encoder"]),
          "-- wrong file or key? Fix ENC before trusting anything below.")

In [ ]:
t0 = time.time()
BANK = RankBank()
for name in ENC:
    tf = BANK.add(name, E[name])
    print(f"  ranked {name:<10} ties={tf:.1e}  ({time.time()-t0:.0f}s)")

repro, VERIFIED = reproduction_gate(BANK, tol=0.01)
print("\nREPRODUCTION GATE (published E.12 / Table C12m values, tol 0.01)")
print(pd.DataFrame(repro).to_string(index=False))
xm = [BANK.rho(a, b) for a in IMG_ENC for b in TXT_ENC]
print(f"\ncross-modal mean over the 16 image-text pairs: {np.mean(xm):.3f}  (published 8-encoder baseline 0.290)")
print("\nVERIFIED -- the pipeline reproduces the report" if VERIFIED else
      "\n!! NOT REPRODUCED -- every verdict below is stamped UNVERIFIED")
STAMP = "" if VERIFIED else "UNVERIFIED (reproduction gate failed) - "

## 7 — The image list: exact COCO ids, in cache order

ConvNeXt's cache stores **real COCO ids** in `keep` (the G4 namespace, E.15), in the same order
as every other cache. Those ids *are* the item list. They must be strictly increasing (the loop
sorts them) and the DINOv2 `keep` must be `0…N−1` (request positions).

In [ ]:
ids = KEEP["convnext"]
if ids is None:
    raise RuntimeError("ConvNeXt cache has no 'keep' array; reconstruct ids = sorted(set(caps) & set(url))[:N] "
                       "from captions_train2017.json as in E.15 and set ids manually")
ids = np.asarray(ids).astype(np.int64)
assert len(ids) == N_ITEMS, len(ids)
assert np.all(np.diff(ids) > 0), "ids not strictly increasing - not the sorted loop order"
if KEEP.get("img_small") is not None:
    assert np.array_equal(KEEP["img_small"], np.arange(N_ITEMS)), "DINOv2 keep is not the request position"
print(f"{len(ids)} COCO ids, {ids[0]} ... {ids[-1]}  (published range 9 ... 46492)")

In [ ]:
CROPS_PATH = WORK / f"crops_{N_ITEMS}.npy"
if SYNTHETIC:
    print("synthetic - no images")
else:
    t0 = time.time()
    status, failed, splits = download_images(ids, WORK / "coco", workers=32)
    print(f"download: {splits}  ({time.time()-t0:.0f}s)")
    if failed:
        raise RuntimeError(f"{len(failed)} images failed, e.g. {list(failed.items())[:3]}. "
                           "Re-run this cell (it resumes). Never continue with holes.")
    if not CROPS_PATH.exists():
        build_crop_cache(ids, WORK / "coco", CROPS_PATH, workers=8)
    CROPS = np.load(CROPS_PATH, mmap_mode="r")
    print("crops:", CROPS.shape, CROPS.dtype, f"({time.time()-t0:.0f}s total)")

## 8 — Fractal checkpoints and the normalization they were trained with

Weights come from the authors' Drive folders linked in their READMEs. The normalization is
**read from each repo's own code**, every declaration found is printed, and the pretraining one
is used. Cell 11 re-extracts with ImageNet statistics as a sensitivity check.

If `gdown` hits a Drive quota: open the folder link, download the FractalDB-**1k** file, put it in
`DATA_DIR/fractal_weights/`, re-run.

In [ ]:
import re

def sha256(p, chunk=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()[:16]

def pick(paths, must, prefer=()):
    """Match on the whole path, not just the file name: some releases nest
    'checkpoint.pth' inside a descriptively named folder."""
    s = lambda p: str(p).lower()
    c = [p for p in paths if all(m.lower() in s(p) for m in must)]
    c.sort(key=lambda p: -sum(k.lower() in s(p) for k in prefer))
    return c[0] if c else None

CKPT, NORM = {}, {}
if not SYNTHETIC:
    wdir = DATA_DIR / "fractal_weights"; wdir.mkdir(exist_ok=True)
    for folder, sub in [(RESNET_FOLDER, "resnet"), (VIT_FOLDER, "vit")]:
        tgt = WORK / f"w_{sub}"
        if not any(tgt.rglob("*.pth*")):
            try:
                gdown.download_folder(folder, output=str(tgt), quiet=True, use_cookies=False)
            except Exception as e:
                print(f"  gdown {sub} failed: {type(e).__name__}: {e}  -- falling back to {wdir}")
    allw = [p for p in list(WORK.rglob("*.pth*")) + list(wdir.rglob("*.pth*")) if p.is_file()]
    print("checkpoints found:"); [print("   ", p.relative_to(p.parents[1]), f"{p.stat().st_size/1e6:.0f} MB") for p in allw]

    def has(pth, *toks):
        sl = str(pth).lower()
        return any(t.lower() in sl for t in toks)

    # frac_cnn: the single FractalDB-1k ResNet-50 (files are named res50, not resnet50;
    # exclude the 10k models and the 20-member ensemble)
    cnn = [pp for pp in allw if has(pp, "res50", "resnet50") and has(pp, "fractal1k", "_1k", "-1k", "1000")
           and not has(pp, "10000", "10k") and "ensemble" not in str(pp).lower()]
    cnn.sort(key=lambda pp: (bool(re.search(r"_res50_\d+", pp.name)), len(pp.name)))
    CKPT["frac_cnn"] = Path(FRACTAL_RESNET_PATH) if FRACTAL_RESNET_PATH else (cnn[0] if cnn else None)

    # frac_vit: the FractalDB-1k DeiT (the only fractal ViT released is DeiT-TINY, 192-d);
    # build_deit reads the real width from pos_embed, so tiny/small/base all work
    vit = [pp for pp in allw if has(pp, "deit", "vit") and has(pp, "fractal1k", "_1k", "-1k")
           and not has(pp, "fractal10k", "10k", "10000")]
    vit.sort(key=lambda pp: len(pp.name))
    CKPT["frac_vit"] = Path(FRACTAL_VIT_PATH) if FRACTAL_VIT_PATH else (vit[0] if vit else None)

    for k, p in CKPT.items():
        if p is None or not Path(p).exists():
            raise FileNotFoundError(
                f"{k}: no FractalDB-1k checkpoint found automatically. Set "
                f"{'FRACTAL_RESNET_PATH' if k=='frac_cnn' else 'FRACTAL_VIT_PATH'} in cell 2 to the .pth path.")
        print(f"  {k}: {Path(p).name}  sha256[:16]={sha256(p)}")
    if has(CKPT["frac_cnn"], "10000", "10k") or has(CKPT["frac_vit"], "fractal10k", "10k", "10000"):
        print("  !! a 10k checkpoint was selected -- the two types are then NOT data-matched; say so in the write-up")
    print("  NOTE: the released fractal ViT is DeiT-TINY (~5M params) while the CNN is ResNet-50 (~25M). "
          "Each is bracketed against its OWN architecture's twins, so P1/P2/P3/P5 and R stay valid per "
          "architecture; only the CNN-vs-ViT contrast (X1) also carries a capacity gap -- flagged, not interpreted.")

    for sub, url in [("resnet", "https://github.com/hirokatsukataoka16/FractalDB-Pretrained-ResNet-PyTorch"),
                     ("vit", "https://github.com/nakashima-kodai/FractalDB-Pretrained-ViT-PyTorch")]:
        repo = WORK / f"repo_{sub}"
        if not repo.exists():
            subprocess.run(["git", "clone", "-q", "--depth", "1", url, str(repo)], check=False)
        found = find_normalizations(repo) if repo.exists() else []
        print(f"\n  normalizations declared in {url.split('/')[-1]}:")
        for h in found:
            print(f"     score={h['score']}  {h['file']}: mean={h['mean']} std={h['std']}")
        key = "frac_cnn" if sub == "resnet" else "frac_vit"
        if found and found[0]["score"] > 0:
            NORM[key] = (found[0]["mean"], found[0]["std"], found[0]["file"])
        else:
            NORM[key] = (IMAGENET_MEAN, IMAGENET_STD, "NOT FOUND in repo - ImageNet stats assumed")
        print(f"  -> using for {key}: mean={NORM[key][0]} std={NORM[key][1]}  [{NORM[key][2]}]")
else:
    print("synthetic - skipped")

## 9 — Model builders with strict loading

A checkpoint that silently fails to load leaves **random weights** — which would masquerade as
the random twin. So loading is strict: after dropping the classifier head, every backbone key must
load (only `num_batches_tracked` may be absent), nothing may be unexpected, and a parameter tensor
must differ from a fresh random init.

In [ ]:
def _torch():
    import torch
    torch.backends.cuda.matmul.allow_tf32 = False      # full fp32: this is a measurement, not a benchmark
    torch.backends.cudnn.allow_tf32 = False
    return torch

def load_strict(model, sd, head_prefixes):
    torch = _torch()
    sd = {k: v for k, v in sd.items() if not k.startswith(head_prefixes)}
    ref = {k: v.detach().clone() for k, v in model.state_dict().items()}
    res = model.load_state_dict(sd, strict=False)
    missing = [k for k in res.missing_keys if not k.endswith("num_batches_tracked")]
    if missing or res.unexpected_keys:
        raise RuntimeError(f"checkpoint does not fit the architecture:\n  missing={missing[:8]} (n={len(missing)})"
                           f"\n  unexpected={res.unexpected_keys[:8]} (n={len(res.unexpected_keys)})")
    changed = sum(not torch.equal(ref[k], model.state_dict()[k]) for k in sd if k in ref and ref[k].dtype.is_floating_point)
    if changed == 0:
        raise RuntimeError("no parameter changed on load -- this would be a random-init model in disguise")
    return len(sd), changed

def read_ckpt(path):
    torch = _torch()
    obj = torch.load(path, map_location="cpu", weights_only=False)   # authors' checkpoints pickle extra objects
    if hasattr(obj, "state_dict") and callable(obj.state_dict):      # a whole pickled nn.Module
        obj = obj.state_dict()
    sd, notes = unwrap_state_dict(obj, torch.is_tensor)
    if not sd:
        raise RuntimeError(f"no tensors found in {path} (top-level keys: {list(obj)[:10]})")
    return sd, notes

def build_resnet50(kind, path=None):
    torch = _torch(); import torchvision
    torch.manual_seed(SEED)
    if kind == "natural":
        m = torchvision.models.resnet50(weights="IMAGENET1K_V1")
        m.fc = torch.nn.Identity(); return m, IMAGENET_MEAN, IMAGENET_STD, "torchvision IMAGENET1K_V1"
    m = torchvision.models.resnet50(weights=None)
    m.fc = torch.nn.Identity()
    if kind == "random":
        return m, IMAGENET_MEAN, IMAGENET_STD, f"random init, seed {SEED}"
    sd, notes = read_ckpt(path)
    n, ch = load_strict(m, sd, ("fc.",))
    return m, None, None, f"{Path(path).name}: {n} tensors loaded, {ch} changed; {notes}"

DEIT = {192: "deit_tiny_patch16_224", 384: "deit_small_patch16_224", 768: "deit_base_patch16_224"}

def build_deit(kind, path=None, width=None):
    torch = _torch()
    torch.manual_seed(SEED)
    if kind == "fractal":
        sd, notes = read_ckpt(path)
        if "dist_token" in sd:
            raise RuntimeError("distilled DeiT checkpoint - the FractalDB ViT is a plain VisionTransformer")
        pe = sd["pos_embed"].shape
        width = pe[-1]
        if pe[1] != 197:
            raise RuntimeError(f"pos_embed {tuple(pe)}: expected 197 tokens (224px, patch 16)")
    name = DEIT[width]
    if kind == "natural":
        tag = f"{name}.fb_in1k"
        try:
            m = timm.create_model(tag, pretrained=True, num_classes=0)
        except Exception:
            m = timm.create_model(name, pretrained=True, num_classes=0); tag = name
        from timm.data import resolve_data_config
        cfg = resolve_data_config({}, model=m)
        return m, tuple(cfg["mean"]), tuple(cfg["std"]), f"timm {tag}", width
    m = timm.create_model(name, pretrained=False, num_classes=0)
    if kind == "random":
        return m, IMAGENET_MEAN, IMAGENET_STD, f"random init {name}, seed {SEED}", width
    n, ch = load_strict(m, sd, ("head.",))
    return m, None, None, f"{Path(path).name} as {name}: {n} tensors loaded, {ch} changed; {notes}", width

def fwd_resnet(m, x):
    return m(x)

def fwd_vit_clspatch(m, x):
    torch = _torch()
    t = m.forward_features(x)                       # timm: final norm already applied
    k = getattr(m, "num_prefix_tokens", 1)
    return torch.cat([t[:, 0], t[:, k:].mean(1)], dim=1)

def extract(model, forward, mean, std, crops, batch=BATCH):
    torch = _torch()
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(dev).eval()
    mu = torch.tensor(mean, device=dev).view(1, 3, 1, 1)
    sd = torch.tensor(std, device=dev).view(1, 3, 1, 1)
    out = []
    with torch.inference_mode():
        for i in range(0, len(crops), batch):
            x = torch.from_numpy(np.ascontiguousarray(crops[i:i + batch])).to(dev)
            x = (x.permute(0, 3, 1, 2).float() / 255.0 - mu) / sd
            out.append(forward(model, x).float().cpu().numpy())
    model.to("cpu")
    if dev == "cuda":
        torch.cuda.empty_cache()
    return np.concatenate(out).astype(np.float32)

print("builders defined")

## 10 — Extract every new space (cached to Drive; re-runs load the cache)

In [ ]:
META = {}
def cached(name, fn):
    f = OUT / f"g9_{name}_{N_ITEMS}.npz"
    if f.exists():
        z = np.load(f, allow_pickle=False)
        assert np.array_equal(z["keep"], ids), f"{name}: cached keep differs from the id list"
        META[name] = json.loads(str(z["meta"]))
        return z["emb"]
    X, meta = fn()
    np.savez(f, emb=X, keep=ids, meta=json.dumps(meta))
    META[name] = meta
    return X

NEW = {}
if SYNTHETIC:
    NEW = {k: v.astype(np.float32) for k, v in SYN_NEW.items()}
    META = {k: {"source": "synthetic"} for k in NEW}
    NEW["frac_cnn_inorm"] = NEW["frac_cnn"] + 0.02 * np.random.default_rng(1).normal(size=NEW["frac_cnn"].shape)
    NEW["frac_vit_inorm"] = NEW["frac_vit"] + 0.02 * np.random.default_rng(2).normal(size=NEW["frac_vit"].shape)
    NEW["rand_cnn_fnorm"] = NEW["rand_cnn"] + 0.03 * np.random.default_rng(3).normal(size=NEW["rand_cnn"].shape)
else:
    def run_resnet(kind, norm=None):
        def f():
            m, mu, sd, desc = build_resnet50(kind, CKPT.get("frac_cnn"))
            if kind == "fractal":
                mu, sd = norm[0], norm[1]
            t = time.time(); X = extract(m, fwd_resnet, mu, sd, CROPS)
            return X, dict(model="resnet50", kind=kind, desc=desc, mean=list(mu), std=list(sd),
                           pooling="global avg 2048", secs=round(time.time() - t))
        return f
    VIT_WIDTH = {}
    def run_deit(kind, norm=None):
        def f():
            m, mu, sd, desc, w = build_deit(kind, CKPT.get("frac_vit"), VIT_WIDTH.get("w"))
            VIT_WIDTH["w"] = w
            if kind == "fractal":
                mu, sd = norm[0], norm[1]
            t = time.time(); X = extract(m, fwd_vit_clspatch, mu, sd, CROPS)
            return X, dict(model=DEIT[w], kind=kind, desc=desc, mean=list(mu), std=list(sd),
                           pooling="cls+patch (cls = first half)", secs=round(time.time() - t))
        return f

    NEW["frac_cnn"] = cached("frac_cnn", run_resnet("fractal", NORM["frac_cnn"]))
    NEW["frac_vit"] = cached("frac_vit", run_deit("fractal", NORM["frac_vit"]))
    VIT_WIDTH["w"] = NEW["frac_vit"].shape[1] // 2          # twins use the fractal ViT's width
    NEW["rand_cnn"] = cached("rand_cnn", run_resnet("random"))
    NEW["rand_vit"] = cached("rand_vit", run_deit("random"))
    NEW["nat_cnn"]  = cached("nat_cnn", run_resnet("natural"))
    NEW["nat_vit"]  = cached("nat_vit", run_deit("natural"))
    NEW["pixels"]   = cached("pixels", lambda: (pixel_features(CROPS), dict(model="pixels 32x32 RGB, centred")))
    for k in ("frac_cnn", "frac_vit"):                      # normalization sensitivity
        if tuple(np.round(NORM[k][0], 3)) != tuple(np.round(IMAGENET_MEAN, 3)):
            fn = run_resnet("fractal", (IMAGENET_MEAN, IMAGENET_STD)) if k == "frac_cnn" else \
                 run_deit("fractal", (IMAGENET_MEAN, IMAGENET_STD))
            NEW[f"{k}_inorm"] = cached(f"{k}_inorm", fn)
    # PREPROCESSING CONFOUND CONTROL: random ResNet-50 weights + FractalDB normalization.
    # rand_cnn uses ImageNet norm; frac_cnn uses repo norm (mean=0.2, std=0.5). So P2-cnn
    # confounds training with preprocessing. This cache isolates the difference: if
    # rand_cnn_fnorm differs materially from rand_cnn, the norm matters and P2-cnn is confounded.
    if "frac_cnn" in NORM and tuple(np.round(NORM["frac_cnn"][0], 3)) != tuple(np.round(IMAGENET_MEAN, 3)):
        def _rand_fnorm():
            m, _, _, desc = build_resnet50("random")
            mu, sd = NORM["frac_cnn"][0], NORM["frac_cnn"][1]
            t = time.time(); X = extract(m, fwd_resnet, mu, sd, CROPS)
            return X, dict(model="resnet50", kind="random", desc=desc + " [FractalDB norm]",
                           mean=list(mu), std=list(sd), pooling="global avg 2048",
                           secs=round(time.time() - t))
        NEW["rand_cnn_fnorm"] = cached("rand_cnn_fnorm", _rand_fnorm)
    NEW["frac_vit_cls"] = NEW["frac_vit"][:, :NEW["frac_vit"].shape[1] // 2]

for k, X in NEW.items():
    print(f"  {k:<16} {str(X.shape):<16} {META.get(k, {}).get('desc', META.get(k, {}).get('model', ''))}")
PRIMARY = ["pixels", "rand_cnn", "frac_cnn", "nat_cnn", "rand_vit", "frac_vit", "nat_vit"]

## 11 — Alignment gate and geometry of the new spaces

Each new space must agree with ConvNeXt (the natural-image CNN already in the roster) far above
its own row-shuffled value, or its rows are not the same images. Geometry flags collapse
(mean pair-cosine) and reports algebraic and entropy rank separately.

In [ ]:
gate, geo = [], []
for k in NEW:
    intact = subsample_rhos(NEW[k], E["convnext"], m=min(1500, N_ITEMS), B=3, seed=SEED)
    shuf = subsample_rhos(NEW[k], E["convnext"], m=min(1500, N_ITEMS), B=3, seed=SEED, shuffle=True)
    ratio = abs(intact.mean()) / max(abs(shuf.mean()), 1e-3)
    gate.append(dict(space=k, rho_vs_convnext=round(intact.mean(), 4), shuffled=round(shuf.mean(), 4),
                     ratio=round(ratio, 1), aligned=ratio >= 5))
    geo.append(dict(space=k, **{a: (round(b, 3) if isinstance(b, float) else b) for a, b in geometry(NEW[k]).items()}))
for k in ENC:
    geo.append(dict(space=k, **{a: (round(b, 3) if isinstance(b, float) else b) for a, b in geometry(E[k]).items()}))
gate, geo = pd.DataFrame(gate), pd.DataFrame(geo)
print(gate.to_string(index=False)); print(); print(geo.to_string(index=False))
bad = list(gate.loc[~gate.aligned, "space"])
if bad:
    print("\n!! alignment gate failed for", bad, "- check the id list and crop order before reading on")
coll = set(geo.loc[geo.mean_pair_cos > 0.9, "space"])
print("\ncollapsed by mean pair-cosine > 0.9:", coll or "none",
      "| documented degenerate:", DEGENERATE)

## 12 — Full-n Spearman: every new space against each of the eight encoders

In [ ]:
t0 = time.time()
for k in NEW:
    BANK.add(k, NEW[k])
    print(f"  ranked {k:<16} ({time.time()-t0:.0f}s)")

RHO = pd.DataFrame({e: {k: BANK.rho(k, e) for k in NEW} for e in ENC})
RHO_NEW = pd.DataFrame({a: {b: BANK.rho(a, b) for b in NEW} for a in NEW})
print("\nrho(new space, encoder) - all items, no map")
print(RHO.loc[PRIMARY + [k for k in NEW if k not in PRIMARY]].to_string(float_format=lambda x: f"{x:6.3f}"))
print("\nConvNeXt row measured in this run (the reproduction gate compares it with Table C12m):",
      {e: round(BANK.rho('convnext', e), 3) for e in ENC if e != 'convnext'})

### 12b — Heatmaps: every new space vs the 8 encoders, and the new-space cross-matrix

Left: raw Spearman rho of each bracket space (rows) against the 8 encoders (columns) — the whole
G9 result in one panel, ImageNet rows bright, fractal rows dim, GPT-2 column marked. Right: the
new spaces against each other.

In [ ]:
import matplotlib.pyplot as plt
rows_order = PRIMARY + [k for k in NEW if k not in PRIMARY]
RHOm = RHO.loc[rows_order, list(ENC)].values.astype(float)
fig, axes = plt.subplots(1, 2, figsize=(19, 7), gridspec_kw={"width_ratios": [1.15, 1]})
im0 = axes[0].imshow(RHOm, cmap="viridis", vmin=0, vmax=max(0.5, np.nanmax(RHOm)), aspect="auto")
axes[0].set_xticks(range(len(ENC)), [e + (" *" if e in DEGENERATE else "") for e in ENC], rotation=45, ha="right", fontsize=9)
axes[0].set_yticks(range(len(rows_order)), rows_order, fontsize=9)
for i in range(len(rows_order)):
    for j in range(len(ENC)):
        v = RHOm[i, j]
        axes[0].text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7, color="w" if v < 0.55 * np.nanmax(RHOm) else "k")
axes[0].axvline(len(IMG_ENC) - 0.5, color="w", lw=1.2)
axes[0].set_title("rho(new space, encoder) — raw, all items, no map")
fig.colorbar(im0, ax=axes[0], fraction=0.046)

NEWm = RHO_NEW.loc[rows_order, rows_order].values.astype(float)
im1 = axes[1].imshow(NEWm, cmap="magma", vmin=0, vmax=1.0, aspect="auto")
axes[1].set_xticks(range(len(rows_order)), rows_order, rotation=45, ha="right", fontsize=8)
axes[1].set_yticks(range(len(rows_order)), rows_order, fontsize=8)
for i in range(len(rows_order)):
    for j in range(len(rows_order)):
        axes[1].text(j, i, f"{NEWm[i,j]:.2f}", ha="center", va="center", fontsize=6, color="w" if NEWm[i,j] < 0.6 else "k")
axes[1].set_title("new spaces vs each other")
fig.colorbar(im1, ax=axes[1], fraction=0.046)
fig.suptitle("G9 raw shape agreement (* GPT-2 collapsed, stratified out of means)", y=1.01)
fig.tight_layout(); fig.savefig(OUT / "G9_rho_heatmaps.png", dpi=150, bbox_inches="tight"); plt.show()

## 13 — Stability over items, and the shuffle null

Every subsample uses the **same** items in every space, so between-model gaps are paired.
This is a stability interval over which 2,000 images were drawn, not a full-n CI.

In [ ]:
rng = np.random.default_rng(SEED)
spaces = {**{k: NEW[k] for k in PRIMARY}, **E}
REPS = {(a, e): [] for a in PRIMARY for e in ENC}
NULL = {(a, e): [] for a in ("frac_cnn", "frac_vit") for e in ENC}
t0 = time.time()
for b in range(SUB_B):
    idx = rng.choice(N_ITEMS, SUB_M, replace=False)
    rk = {k: standardized_ranks(upper_tri_cos(X, idx))[0].astype(np.float64) for k, X in spaces.items()}
    perm = rng.permutation(idx)
    for a in PRIMARY:
        for e in ENC:
            REPS[(a, e)].append(float(rk[a] @ rk[e]))
    for a in ("frac_cnn", "frac_vit"):
        ra = standardized_ranks(upper_tri_cos(NEW[a], perm))[0].astype(np.float64)
        for e in ENC:
            NULL[(a, e)].append(float(ra @ rk[e]))
    if (b + 1) % 5 == 0:
        print(f"  {b+1}/{SUB_B} subsamples ({time.time()-t0:.0f}s)")
REPS = {k: np.array(v) for k, v in REPS.items()}
NULL = {k: np.array(v) for k, v in NULL.items()}
sd_tab = pd.DataFrame({e: {a: REPS[(a, e)].std(ddof=1) for a in PRIMARY} for e in ENC})
print("\nsubsample SD of rho:"); print(sd_tab.to_string(float_format=lambda x: f"{x:6.4f}"))
print("\nshuffle null (mean):", {k: round(v.mean(), 4) for k, v in NULL.items() if k[1] in ("img_base", "bge")})

## 14 — Brackets, reality fraction, and the pre-registered verdicts

In [ ]:
ARCH = {"cnn": ("rand_cnn", "frac_cnn", "nat_cnn"), "vit": ("rand_vit", "frac_vit", "nat_vit")}
HEALTHY_TXT = [e for e in TXT_ENC if e not in DEGENERATE]

Rtab, Rrep = {}, {}
for arch, (r, f, n) in ARCH.items():
    for e in ENC:
        Rtab[(arch, e)] = reality_fraction(RHO.loc[f, e], RHO.loc[r, e], RHO.loc[n, e], MIN_BRACKET)
        Rrep[(arch, e)] = np.array([reality_fraction(a, b, c, MIN_BRACKET) for a, b, c in
                                    zip(REPS[(f, e)], REPS[(r, e)], REPS[(n, e)])])
R_df = pd.DataFrame({e: {arch: Rtab[(arch, e)] for arch in ARCH} for e in ENC})
print("Reality fraction R (0 = architecture only, 1 = as good as ImageNet); NaN = bracket < 0.02")
print(R_df.to_string(float_format=lambda x: f"{x:6.3f}"))

V, detail = {}, {}
# item-level paired bootstrap pool: one fixed working sample of items, resampled with
# replacement inside paired_item_bootstrap_rho. This gives an HONEST CI for the R-difference,
# replacing the invalid bootstrap of 30 overlapping subsample means.
_P3_ENC = IMG_ENC + [e for e in TXT_ENC if e not in DEGENERATE]   # encoders the diff uses
_boot_pool = np.random.default_rng(SEED).choice(N_ITEMS, min(P3_POOL, N_ITEMS), replace=False)
_P3B = paired_item_bootstrap_rho({k: NEW[k] for k in ("frac_cnn","frac_vit","rand_cnn","rand_vit","nat_cnn","nat_vit")},
                                 {e: E[e] for e in _P3_ENC}, _boot_pool, B=P3_BOOT, seed=SEED)
def _R_reps(f, r, n, e):
    num = _P3B[(f, e)] - _P3B[(r, e)]; den = _P3B[(n, e)] - _P3B[(r, e)]
    out = num / den; out[np.abs(den) < MIN_BRACKET] = np.nan
    return out
for arch, (r, f, n) in ARCH.items():
    p1 = [gap_exceeds(REPS[(n, e)], REPS[(f, e)], GAP_K) for e in ENC]
    p2 = [gap_exceeds(REPS[(f, e)], REPS[(r, e)], GAP_K) for e in IMG_ENC]
    p5 = [gap_exceeds(REPS[(f, e)], REPS[("pixels", e)], GAP_K) for e in IMG_ENC]
    V[("P1", arch)] = (verdict(sum(g for g, _, _ in p1), len(ENC), 7), sum(g for g, _, _ in p1), len(ENC))
    V[("P2", arch)] = (verdict(sum(g for g, _, _ in p2), 4, 3), sum(g for g, _, _ in p2), 4)
    V[("P5", arch)] = (verdict(sum(g for g, _, _ in p5), 4, 3), sum(g for g, _, _ in p5), 4)
    # per-replicate img-vs-text R difference, properly paired over the SAME item resample
    img_reps = np.nanmean(np.stack([_R_reps(f, r, n, e) for e in IMG_ENC]), 0)
    txt_reps = np.nanmean(np.stack([_R_reps(f, r, n, e) for e in HEALTHY_TXT]), 0)
    diff = img_reps - txt_reps
    diff = diff[np.isfinite(diff)]
    if diff.size < 30:
        V[("P3", arch)] = ("NOT ESTIMABLE", np.nan, np.nan); lo = hi = np.nan
    else:
        lo, hi = np.percentile(diff, [2.5, 97.5])
        V[("P3", arch)] = ("CONFIRMED" if lo > 0 else ("FALSIFIED" if hi < 0 else "INCONCLUSIVE"),
                           float(lo), float(hi))
    detail[arch] = dict(img_R=float(np.nanmean([Rtab[(arch, e)] for e in IMG_ENC])),
                        txt_R=float(np.nanmean([Rtab[(arch, e)] for e in HEALTHY_TXT])),
                        diff_lo=float(lo), diff_hi=float(hi),
                        gpt2_R=Rtab[(arch, "gpt2")])

print("\nPRE-REGISTERED VERDICTS" + (" -- UNVERIFIED: reproduction gate failed" if not VERIFIED else ""))
for (p, arch), v in sorted(V.items()):
    if p != "P3":
        extra = f"{v[1]}/{v[2]}"
    elif not np.isfinite(v[1]):
        extra = "R_img - R_txt CI not estimable at this held-out size"
    else:
        extra = f"R_img - R_txt 95% interval [{v[1]:+.3f}, {v[2]:+.3f}]"
    print(f"  {p} [{arch}]  {STAMP}{v[0]:<13} {extra}")
for arch, d in detail.items():
    print(f"  R [{arch}]: image mean {d['img_R']:.3f}, healthy-text mean {d['txt_R']:.3f}, GPT-2 {d['gpt2_R']:.3f} (stratified out)")

# X1 exploratory and sensitivity
x1 = {e: (Rtab[("cnn", e)], Rtab[("vit", e)]) for e in ENC}
sens = {}
if "frac_vit_cls" in NEW:
    sens["ViT cls-only vs cls+patch"] = float(np.max(np.abs(RHO.loc["frac_vit_cls"] - RHO.loc["frac_vit"])))
for k in ("frac_cnn", "frac_vit"):
    if f"{k}_inorm" in NEW:
        sens[f"{k}: ImageNet vs repo normalization"] = float(np.max(np.abs(RHO.loc[f"{k}_inorm"] - RHO.loc[k])))
# PREPROCESSING CONFOUND CHECK: does changing the norm on RANDOM weights change rho?
# If rand_cnn_fnorm differs from rand_cnn, then P2-cnn confounds training with preprocessing.
if "rand_cnn_fnorm" in NEW:
    d = float(np.max(np.abs(RHO.loc["rand_cnn_fnorm"] - RHO.loc["rand_cnn"])))
    sens["rand_cnn: ImageNet vs FractalDB normalization"] = d
print("\nSENSITIVITY (max |delta rho| over the 8 encoders; flag > 0.02):")
for k, v in sens.items():
    print(f"  {k:<48} {v:.4f}  {'FLAG' if v > 0.02 else 'ok'}")
if "rand_cnn_fnorm" in NEW:
    # print the confound assessment
    d_rand = sens.get("rand_cnn: ImageNet vs FractalDB normalization", 0)
    d_frac = sens.get("frac_cnn: ImageNet vs repo normalization", 0)
    # if the norm effect on random weights is comparable to the fractal signal, P2-cnn is confounded
    frac_signal = float(np.mean([RHO.loc["frac_cnn", e] - RHO.loc["rand_cnn", e] for e in IMG_ENC]))
    print(f"\n  CONFOUND CHECK: norm changes random-CNN rho by up to {d_rand:.4f}")
    print(f"    fractal-over-random signal (image-encoder mean): {frac_signal:.4f}")
    if d_rand > abs(frac_signal) * 0.5:
        print(f"    WARNING: norm effect ({d_rand:.4f}) is >= 50% of the fractal signal ({frac_signal:.4f})")
        print(f"    -> P2-cnn is CONFOUNDED by preprocessing. Report both rand_cnn norms.")
    else:
        print(f"    norm effect is < 50% of the fractal signal -> preprocessing confound is minor")
    print(f"    rand_cnn (ImageNet norm): image-enc mean rho = {np.mean([RHO.loc['rand_cnn', e] for e in IMG_ENC]):.4f}")
    print(f"    rand_cnn_fnorm (FractalDB norm): image-enc mean rho = {np.mean([RHO.loc['rand_cnn_fnorm', e] for e in IMG_ENC]):.4f}")
    print(f"    frac_cnn (FractalDB norm): image-enc mean rho = {np.mean([RHO.loc['frac_cnn', e] for e in IMG_ENC]):.4f}")

## 15 — Figure: where the fractal models land, per encoder

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5), sharey=True)
order = IMG_ENC + TXT_ENC
cols = {"pixels": "#bdbdbd", "rand": "#9ecae1", "frac": "#e6550d", "nat": "#31a354"}
for ax, (arch, (r, f, n)) in zip(axes, ARCH.items()):
    x = np.arange(len(order)); w = 0.2
    for j, (lab, key) in enumerate([("pixels 32x32", "pixels"), ("random init", r), ("FractalDB-1k", f), ("ImageNet-1k", n)]):
        mu = [RHO.loc[key, e] for e in order]
        er = [REPS[(key, e)].std(ddof=1) for e in order]
        ax.bar(x + (j - 1.5) * w, mu, w, yerr=er, capsize=2, label=lab,
               color=cols[key.split("_")[0]] if key != "pixels" else cols["pixels"])
    ax.axvline(len(IMG_ENC) - 0.5, color="k", lw=0.8, ls=":")
    ax.set_xticks(x, [e + (" *" if e in DEGENERATE else "") for e in order], rotation=30)
    ax.set_title(f"{'ResNet-50' if arch=='cnn' else 'DeiT'}: Spearman shape agreement with each encoder")
    ax.text(1.5, ax.get_ylim()[1] * 0.95, "image encoders", ha="center"); ax.text(5.5, ax.get_ylim()[1] * 0.95, "text encoders", ha="center")
axes[0].set_ylabel("rho (all items, no map)")
fig.legend(*axes[0].get_legend_handles_labels(), loc="lower center", ncol=4, fontsize=9, bbox_to_anchor=(0.5, -0.08))
fig.text(0.5, -0.12, "* GPT-2: documented collapsed space, stratified out of every mean. Error bars: SD over "
         f"{SUB_B} subsamples of {SUB_M} items.", ha="center", fontsize=8)
fig.tight_layout(); fig.savefig(OUT / "G9_fractal_brackets.png", dpi=150, bbox_inches="tight"); plt.show()

## 16 — The PRH evidence table, with the G9 rows filled from this run

Every measure in the project, ordered by the PRH proof step it serves. Existing rows are
transcribed from the report (source of truth); the S1b rows are written from the numbers above.

In [ ]:
fmt = lambda k: ", ".join(f"{e} {RHO.loc[k, e]:.3f}" for e in ENC)
def r_str(v):
    return "undefined (bracket < 0.02)" if not np.isfinite(v) else f"{v:.3f}"

def ci_str(lo, hi):
    return "[not estimable]" if not np.isfinite(lo) else f"[{lo:+.3f}, {hi:+.3f}]"

def row_rho(arch):
    r, f, n = ARCH[arch]
    v, c, t = V[("P1", arch)]
    return (f"{STAMP}FRACTAL: {fmt(f)}. | ImageNet twin: {fmt(n)}. | P1 {v} ({c}/{t} encoders below the "
            f"ImageNet twin by > {GAP_K:g} SD)")

g9 = {
    "F1": row_rho("cnn"),
    "F1c": f"Random twin: {fmt('rand_cnn')}. Shuffle null mean {np.mean([NULL[('frac_cnn', e)].mean() for e in ENC]):+.4f}",
    "F1p": STAMP + ("Fractal training stops short of what natural images give on nearly every encoder: the convergence "
            "is data-driven, as PRH's mechanism says" if V[("P1", "cnn")][0] == "CONFIRMED" else
            "Fractal-only training reaches natural-image agreement on too many encoders: training on any structured "
            "data suffices here, which weakens the 'reality' reading of PRH"),
    "F2": row_rho("vit"),
    "F2c": f"Random twin: {fmt('rand_vit')}. Shuffle null mean {np.mean([NULL[('frac_vit', e)].mean() for e in ENC]):+.4f}",
    "F2p": STAMP + ("Same P1 verdict on a transformer - not a CNN idiosyncrasy" if V[("P1", "vit")][0] == V[("P1", "cnn")][0]
            else "The two architectures DISAGREE on P1 - the answer depends on architecture; report both, claim neither"),
    "F3": (f"{STAMP}Pixels (image-encoder mean) {np.mean([RHO.loc['pixels', e] for e in IMG_ENC]):.3f}; random CNN/ViT "
           f"{np.mean([RHO.loc['rand_cnn', e] for e in IMG_ENC]):.3f}/{np.mean([RHO.loc['rand_vit', e] for e in IMG_ENC]):.3f}; "
           f"fractal {np.mean([RHO.loc['frac_cnn', e] for e in IMG_ENC]):.3f}/{np.mean([RHO.loc['frac_vit', e] for e in IMG_ENC]):.3f}; "
           f"ImageNet {np.mean([RHO.loc['nat_cnn', e] for e in IMG_ENC]):.3f}/{np.mean([RHO.loc['nat_vit', e] for e in IMG_ENC]):.3f}. "
           f"P2 cnn {V[('P2','cnn')][0]} ({V[('P2','cnn')][1]}/4), vit {V[('P2','vit')][0]} ({V[('P2','vit')][1]}/4); "
           f"P5 cnn {V[('P5','cnn')][0]}, vit {V[('P5','vit')][0]}"),
    "F4c": f"Subsample interval ({SUB_B} x {SUB_M:,} items); R undefined where the bracket < {MIN_BRACKET}",
    "F4": STAMP + "; ".join(f"[{a}] R image {d['img_R']:.3f}, healthy text {d['txt_R']:.3f}, "
                    f"R_img-R_txt 95% {ci_str(d['diff_lo'], d['diff_hi'])} -> P3 {V[('P3', a)][0]}; "
                    f"GPT-2 R {r_str(d['gpt2_R'])} (stratified out)"
                    for a, d in detail.items()),
    "F4p": STAMP + ("Fractals supply generic visual structure, but the part shared with LANGUAGE needs natural images - "
            "the most PRH-specific result here" if all(V[("P3", a)][0] == "CONFIRMED" for a in ARCH) else
            "No reliable image-vs-text difference in R: the language-shared structure is not specially dependent "
            "on natural-image training at this scale (or the brackets are too narrow to tell)"),
    "F5": "; ".join(f"{k}: max |d-rho| {v:.4f}{' FLAG' if v > 0.02 else ''}" for k, v in sens.items()) or "not run",
    # F.6 stays PENDING here; the whitening cells (17-18) rewrite the table with it filled
}
TABLE_DF = build_table(g9)
TABLE_DF.to_csv(OUT / "PRH_Evidence_Table.csv", index=False)
write_xlsx(TABLE_DF, OUT / "PRH_Evidence_Table.xlsx", "PRH evidence table - with G9 fractal results")
try:
    (OUT / "PRH_Evidence_Table.md").write_text(TABLE_DF.to_markdown(index=False))
except ImportError:                                   # to_markdown needs 'tabulate'
    (OUT / "PRH_Evidence_Table.md").write_text(TABLE_DF.to_string(index=False))
pd.set_option("display.max_colwidth", 400)
from IPython.display import display, HTML
display(HTML(TABLE_DF.to_html(index=False).replace("<td>", '<td style="vertical-align:top;font-size:11px">')))
json.dump(dict(verified=VERIFIED, repro=repro, verdicts={f"{p}_{a}": list(map(str, v)) for (p, a), v in V.items()},
               R={f"{a}_{e}": (None if not np.isfinite(v) else v) for (a, e), v in Rtab.items()},
               rho=RHO.round(5).to_dict(), sensitivity=sens, checkpoints={k: str(v) for k, v in CKPT.items()},
               normalization={k: [list(v[0]), list(v[1]), v[2]] for k, v in NORM.items()}, meta=META,
               n_items=N_ITEMS, sub=[SUB_M, SUB_B], synthetic=SYNTHETIC),
          open(OUT / "G9_results.json", "w"), indent=2, default=str)
print("written:", *sorted(p.name for p in OUT.iterdir() if p.suffix in (".csv", ".xlsx", ".md", ".json", ".png")), sep="\n  ")

## 17 — De-anisotropization diagnostic (the C.11 move): held-out, five-view ablation

`frac_cnn` has mean pair-cosine **0.85** — **highly anisotropic** (nearly all vectors point one
way), though not fully collapsed: its effective rank is still ~368, unlike genuinely collapsed
GPT-2 (0.999, eff-rank 6.6) or `rand_cnn` (0.998). Raw Spearman ρ on such a space measures the
residual after anisotropy dominates, which is why raw `frac_cnn` sits at the pixel floor. C.11's
move — remove the anisotropy, then re-measure — applies, and is legitimate because the row
correspondence is **known** (positional): the *reading* regime, not the *finding* regime that
whitening destroys (C.12).

Three refinements over the first pass, following review:

1. **Five views, not three**, to separate which operation moves the ordering — raw / **centered**
   (mean only) / **drop-1** (centered + PC1 removed) / **PCA-256** (centered + truncated, variance
   *kept*) / **whitened-256** (centered + truncated + variance *equalized*). The earlier `drop_top_k`
   conflated centering with PC removal; `centered` now isolates it.
2. **Held-out.** Every transform is *fit on the 8,533 train rows* and *applied to the 1,000 held-out
   rows*; ρ is measured on the held-out rows only. So the de-anisotropization must generalize, not
   just describe the sample it was estimated on.
3. Wording: this **exposes a small residual signal**, it does not "rescue" the CNN — absolute ρ
   falls under whitening (the controls fall further). It is **not** the Atlas sense of *recoverable*
   (a fitted entry-map carrying a frozen head); that test is logged as future work.

Secondary, labelled diagnostic. The pre-registered raw verdicts above stand unchanged.

In [ ]:
WHITEN_DIM = 256

# hold-out split: fit transforms on TRAIN rows, measure agreement on HELD-OUT rows
rng_split = np.random.default_rng(SEED)
perm = rng_split.permutation(N_ITEMS)
TR = np.sort(perm[:8533]) if N_ITEMS >= 9533 else np.sort(perm[:int(0.9 * N_ITEMS)])
HO = np.sort(np.setdiff1d(np.arange(N_ITEMS), TR))
print(f"fit transforms on {len(TR)} train rows, measure rho on {len(HO)} held-out rows")

def _fit_pca(Xtr, k, eps=1e-6):
    mu = Xtr.mean(0, keepdims=True)
    U, S, Vt = np.linalg.svd(Xtr - mu, full_matrices=False)
    k = min(k, int((S > eps * S[0]).sum()))
    return mu, Vt[:k], S[:k], len(Xtr)

def make_views(X, k=WHITEN_DIM):
    """Return the five views of X, each fit on TRAIN rows and returned on HELD-OUT rows."""
    X = np.asarray(X, dtype=np.float64)
    Xtr, Xho = X[TR], X[HO]
    mu, Vk, Sk, ntr = _fit_pca(Xtr, k)
    Xc = Xho - mu                                 # centered held-out rows (train mean)
    proj = Xc @ Vk.T                              # centered held-out rows in the train PCA basis
    return {
        "raw":       Xho.astype(np.float32),
        "centered":  Xc.astype(np.float32),
        # drop1 = CENTERED data with its top-PC projection removed (was: raw minus centered-PC1,
        # which left the mean in and did not match the 'centered, PC1 removed' label). Fixed.
        "drop1":     (Xc - (Xc @ Vk[0:1].T) @ Vk[0:1]).astype(np.float32),
        "pca":       proj.astype(np.float32),                              # truncated, variance kept
        "white":     (proj * (np.sqrt(ntr - 1) / Sk)).astype(np.float32),  # variance equalized
    }

VIEWS_ORDER = ["raw", "centered", "drop1", "pca", "white"]
BRACKET = ["pixels", "rand_cnn", "frac_cnn", "nat_cnn", "rand_vit", "frac_vit", "nat_vit"]
ARCHW = {"cnn": ("rand_cnn", "frac_cnn", "nat_cnn"), "vit": ("rand_vit", "frac_vit", "nat_vit")}
arch_lbl = lambda a: "ResNet-50" if a == "cnn" else "DeiT"
IMG_ENC_ = IMG_ENC  # alias guard
VW_SPACES = {k: make_views(NEW[k]) for k in BRACKET}
ENC_HO = {e: E[e][HO] for e in ENC}               # encoders measured on the same held-out rows

wg = pd.DataFrame([dict(space=k,
                        raw_cos=round(geometry(NEW[k][HO])["mean_pair_cos"], 3),
                        centered_cos=round(geometry(VW_SPACES[k]["centered"])["mean_pair_cos"], 3),
                        pca_cos=round(geometry(VW_SPACES[k]["pca"])["mean_pair_cos"], 3),
                        white_cos=round(geometry(VW_SPACES[k]["white"])["mean_pair_cos"], 3))
                   for k in BRACKET])
print("held-out geometry per view (mean pair-cosine):")
print(wg.to_string(index=False))

In [ ]:
# full held-out rho for each of the five views against every encoder;
# cache encoder ranks once (they don't depend on the view)
_ENC_RANKS = {e: standardized_ranks(upper_tri_cos(ENC_HO[e]))[0].astype(np.float64) for e in ENC}
def rho_view(view):
    rows = {}
    for k in BRACKET:
        a = standardized_ranks(upper_tri_cos(VW_SPACES[k][view]))[0].astype(np.float64)
        rows[k] = {e: float(a @ _ENC_RANKS[e]) for e in ENC}
    return pd.DataFrame(rows).T[list(ENC)]

RHO = {v: rho_view(v) for v in VIEWS_ORDER}
RHO_RAW, RHO_W, RHO_D1 = RHO["raw"], RHO["white"], RHO["drop1"]
for v in VIEWS_ORDER:
    print(f"\n{v.upper()} rho (held-out):")
    print(RHO[v].to_string(float_format=lambda x: f"{x:6.3f}"))

In [ ]:
def mean_img(df, k):
    return float(np.mean([df.loc[k, e] for e in IMG_ENC]))

print("Which operation moves the ordering? frac_cnn image-encoder mean per view (held-out):")
print(f"  {'view':<10}{'frac_cnn':>10}{'pixels':>9}{'random':>9}{'ImageNet':>10}   frac>pixels & frac>random?")
for v in VIEWS_ORDER:
    df = RHO[v]
    fc, px, rc, nc = mean_img(df,'frac_cnn'), mean_img(df,'pixels'), mean_img(df,'rand_cnn'), mean_img(df,'nat_cnn')
    print(f"  {v:<10}{fc:>10.3f}{px:>9.3f}{rc:>9.3f}{nc:>10.3f}   {'YES' if (fc>px and fc>rc) else 'no'}")
print("\nfrac_vit image-encoder mean per view (held-out):")
for v in VIEWS_ORDER:
    df = RHO[v]
    fv, px, rv, nv = mean_img(df,'frac_vit'), mean_img(df,'pixels'), mean_img(df,'rand_vit'), mean_img(df,'nat_vit')
    print(f"  {v:<10} frac {fv:.3f}  (pixels {px:.3f}, random {rv:.3f}, ImageNet {nv:.3f})")
print("\nReading: absolute rho FALLS under whitening; the point is the ORDERING (frac above whitened "
      "pixels/random), and WHICH view first makes it hold. If 'centered' already orders frac>pixels, "
      "the effect is anisotropy in the mean; if only 'white' does, it needs full variance equalization.")

In [ ]:
# held-out subsample stability -> gap tests, on the WHITENED view
rng = np.random.default_rng(SEED)
nho = len(HO)
m_ho = min(SUB_M, nho)
sp = {**{f"{k}__white": VW_SPACES[k]["white"] for k in BRACKET}, **ENC_HO}
REPS_W = {(k, e): [] for k in BRACKET for e in ENC}
for b in range(SUB_B):
    idx = rng.choice(nho, m_ho, replace=False)
    rk = {n: standardized_ranks(upper_tri_cos(X, idx))[0].astype(np.float64) for n, X in sp.items()}
    for k in BRACKET:
        for e in ENC:
            REPS_W[(k, e)].append(float(rk[f"{k}__white"] @ rk[e]))
    if (b + 1) % 10 == 0:
        print(f"  held-out whitened subsamples {b+1}/{SUB_B}")
REPS_W = {k: np.array(v) for k, v in REPS_W.items()}

ARCHW = {"cnn": ("rand_cnn", "frac_cnn", "nat_cnn"), "vit": ("rand_vit", "frac_vit", "nat_vit")}
HEALTHY_TXT = [e for e in TXT_ENC if e not in DEGENERATE]
# item-level paired bootstrap for P3w on the WHITENED held-out spaces (same valid method as raw P3)
_P3W_ENC = IMG_ENC + [e for e in TXT_ENC if e not in DEGENERATE]
_wpool = np.random.default_rng(SEED).choice(len(HO), min(P3_POOL, len(HO)), replace=False)
_P3WB = paired_item_bootstrap_rho({k: VW_SPACES[k]["white"] for k in ("frac_cnn","frac_vit","rand_cnn","rand_vit","nat_cnn","nat_vit")},
                                  {e: ENC_HO[e] for e in _P3W_ENC}, _wpool, B=P3_BOOT, seed=SEED)
def _Rw_reps(f, r, n, e):
    num = _P3WB[(f, e)] - _P3WB[(r, e)]; den = _P3WB[(n, e)] - _P3WB[(r, e)]
    out = num / den; out[np.abs(den) < MIN_BRACKET] = np.nan
    return out
VW, Rw_detail, Rw_tab = {}, {}, {}
for arch, (r, f, n) in ARCHW.items():
    p1 = [gap_exceeds(REPS_W[(n, e)], REPS_W[(f, e)], GAP_K)[0] for e in ENC]
    p2 = [gap_exceeds(REPS_W[(f, e)], REPS_W[(r, e)], GAP_K)[0] for e in IMG_ENC]
    p5 = [gap_exceeds(REPS_W[(f, e)], REPS_W[("pixels", e)], GAP_K)[0] for e in IMG_ENC]
    VW[("P1w", arch)] = (verdict(sum(p1), 8, 7), sum(p1))
    VW[("P2w", arch)] = (verdict(sum(p2), 4, 3), sum(p2))
    VW[("P5w", arch)] = (verdict(sum(p5), 4, 3), sum(p5))
    for e in ENC:
        Rw_tab[(arch, e)] = reality_fraction(RHO_W.loc[f, e], RHO_W.loc[r, e], RHO_W.loc[n, e], MIN_BRACKET)
    img_reps = np.nanmean(np.stack([_Rw_reps(f, r, n, e) for e in IMG_ENC]), 0)
    txt_reps = np.nanmean(np.stack([_Rw_reps(f, r, n, e) for e in HEALTHY_TXT]), 0)
    diff = img_reps - txt_reps
    diff = diff[np.isfinite(diff)]
    if diff.size < 30:
        VW[("P3w", arch)] = ("NOT ESTIMABLE", np.nan, np.nan)
    else:
        lo, hi = np.percentile(diff, [2.5, 97.5])
        VW[("P3w", arch)] = ("CONFIRMED" if lo > 0 else ("FALSIFIED" if hi < 0 else "INCONCLUSIVE"),
                             float(lo), float(hi))
    Rw_detail[arch] = dict(img=float(np.nanmean([Rw_tab[(arch, e)] for e in IMG_ENC])),
                           txt=float(np.nanmean([Rw_tab[(arch, e)] for e in HEALTHY_TXT])))

print("HELD-OUT WHITENED DIAGNOSTIC verdicts (NOT the pre-registered raw test - a separate reading question)"
      + ("" if VERIFIED else " -- UNVERIFIED"))
for (p, arch), v in sorted(VW.items()):
    if p != "P3w":
        extra = f"{v[1]}/{'8' if p=='P1w' else '4'}"
    elif not np.isfinite(v[1]):
        extra = "R_img-R_txt CI not estimable at this held-out size"
    else:
        extra = f"R_img-R_txt 95% [{v[1]:+.3f}, {v[2]:+.3f}]"
    print(f"  {p} [{arch}]  {STAMP}{v[0]:<15} {extra}")
for arch, d in Rw_detail.items():
    print(f"  R_white [{arch}]: image {d['img']:.3f}, healthy-text {d['txt']:.3f}")

# magnitude- and scope-qualified read-out
for arch in ARCHW:
    Ri = Rw_detail[arch]["img"]; p5 = VW[("P5w", arch)][0]; p2 = VW[("P2w", arch)][0]
    size = ("small" if Ri < 0.25 else "moderate" if Ri < 0.6 else "large")
    if p5 == "CONFIRMED" and p2 == "CONFIRMED":
        msg = (f"a {size} residual signal became visible after de-anisotropization "
               f"({Ri:.0%} of the natural-image bracket, held-out)")
    else:
        msg = "no residual signal above whitened pixel/random controls"
    print(f"  RESULT [{arch}]: {STAMP}{msg}; P2w {p2}, P5w {p5}")
print("  -> whitening EXPOSES the ordering (small residual > controls), it does NOT raise absolute rho, "
      "and it is NOT the Atlas sense of recoverable (fitted entry-map + frozen head) -- that is future work.")

## 17b — Whitening-dimension sweep: is the residual-signal ordering stable in k?

`WHITEN_DIM` is not a quality knob. Raising k adds lower-variance (noisier) principal directions and
equalizes them to the same scale, which tends to LOWER rho, not raise it — and a very large k walks
back into the 1/S near-null division that is the width-cliff artifact (E.8). The defensible claim is
not "k=256 is best" but "the fractal > whitened-control ordering holds across k". This sweeps
k in {64, 128, 256, 384, 512}, held-out, and checks exactly that. (frac_vit is DeiT-tiny, 384-d, so
k is capped at 384 for the ViT spaces — reported as eff_k.)

In [ ]:
KS = [64, 128, 256, 384, 512]

# cache one train SVD per space, then slice k (fast); measure on held-out rows
FITS = {}
for sp in BRACKET:
    X = NEW[sp].astype(np.float64)
    mu = X[TR].mean(0, keepdims=True)
    U, S, Vt = np.linalg.svd(X[TR] - mu, full_matrices=False)
    FITS[sp] = (mu, Vt, S, X[HO])

def _white_ranks(sp, k):
    mu, Vt, S, Xho = FITS[sp]
    kk = min(k, int((S > 1e-6 * S[0]).sum()))
    proj = (Xho - mu) @ Vt[:kk].T
    white = (proj * (np.sqrt(len(TR) - 1) / S[:kk])).astype(np.float32)
    return standardized_ranks(upper_tri_cos(white))[0].astype(np.float64), kk

sweep = []
for k in KS:
    rho_k = {}
    effk = {}
    for sp in BRACKET:
        a, kk = _white_ranks(sp, k)
        effk[sp] = kk
        rho_k[sp] = {e: float(a @ _ENC_RANKS[e]) for e in ENC}
    for arch, (r, f, n) in ARCHW.items():
        mi = lambda sp: float(np.mean([rho_k[sp][e] for e in IMG_ENC]))
        fr, px, rc, nc = mi(f), mi("pixels"), mi(r), mi(n)
        sweep.append(dict(k=k, arch=arch, eff_k=effk[f], frac=round(fr, 4), pixels=round(px, 4),
                          random=round(rc, 4), imagenet=round(nc, 4),
                          frac_gt_pixels=fr > px, frac_gt_random=fr > rc, ordering_holds=(fr > px and fr > rc)))
SWEEP = pd.DataFrame(sweep)
print(SWEEP.to_string(index=False))
stable = {a: bool(SWEEP.loc[SWEEP.arch == a, "ordering_holds"].all()) for a in ARCHW}
print("\nordering (fractal > whitened pixels AND > whitened random) holds across ALL k:")
for a, ok in stable.items():
    print(f"  {arch_lbl(a)}: {'YES - conclusion robust to k' if ok else 'NO - k-sensitive, report the sweep'}")

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for ax, arch in zip(axes, ARCHW):
    d = SWEEP[SWEEP.arch == arch]
    ax.plot(d.k, d.frac, "o-", color="#e6550d", label="fractal")
    ax.plot(d.k, d.pixels, "s--", color="#bdbdbd", label="pixels (whitened)")
    ax.plot(d.k, d["random"], "^--", color="#9ecae1", label="random (whitened)")
    ax.plot(d.k, d.imagenet, "d-", color="#31a354", label="ImageNet (whitened)")
    ax.set_xlabel("whitening dimension k"); ax.set_ylabel("held-out image-encoder mean rho")
    ax.set_title(f"{arch_lbl(arch)}: rho vs whitening dim (held-out)")
    ax.set_xticks(KS)
    if arch == "cnn": ax.legend(fontsize=8)
fig.suptitle("Whitening-dimension sweep: the fractal>control ordering is what must be stable, not the absolute rho", y=1.02)
fig.tight_layout(); fig.savefig(OUT / "G9_whiten_dim_sweep.png", dpi=150, bbox_inches="tight"); plt.show()
SWEEP.to_csv(OUT / "G9_whiten_dim_sweep.csv", index=False)
print("saved G9_whiten_dim_sweep.{png,csv}")

### 17c — Heatmap: raw vs whitened held-out rho (bracket spaces x 8 encoders)

In [ ]:
import matplotlib.pyplot as plt
rows = BRACKET
fig, axes = plt.subplots(1, 2, figsize=(18, 6.5))
for ax, (view, tag) in zip(axes, [("raw", "raw"), ("white", "whitened (held-out)")]):
    Mv = RHO[view].loc[rows, list(ENC)].values.astype(float)
    im = ax.imshow(Mv, cmap="viridis", vmin=0, vmax=max(0.3, np.nanmax(Mv)), aspect="auto")
    ax.set_xticks(range(len(ENC)), [e + (" *" if e in DEGENERATE else "") for e in ENC], rotation=45, ha="right", fontsize=9)
    ax.set_yticks(range(len(rows)), rows, fontsize=9)
    for i in range(len(rows)):
        for j in range(len(ENC)):
            ax.text(j, i, f"{Mv[i,j]:.2f}", ha="center", va="center", fontsize=7, color="w" if Mv[i,j] < 0.55 * max(0.3, np.nanmax(Mv)) else "k")
    ax.axvline(len(IMG_ENC) - 0.5, color="w", lw=1.2); ax.set_title(f"held-out rho — {tag}")
    fig.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle("De-anisotropization heatmap: raw vs whitened (absolute rho falls; the ordering is the point)", y=1.02)
fig.tight_layout(); fig.savefig(OUT / "G9_whitening_heatmap.png", dpi=150, bbox_inches="tight"); plt.show()

## 18 — Figure: five-view ablation (held-out), and save

In [ ]:
import matplotlib.pyplot as plt
order = IMG_ENC + TXT_ENC
fig, axes = plt.subplots(2, 2, figsize=(17, 10), sharex="col")
for col, (arch, (r, f, n)) in enumerate(ARCHW.items()):
    for rowi, (view, tag) in enumerate([("raw", "raw"), ("white", "whitened (held-out)")]):
        ax = axes[rowi, col]; df = RHO[view]; x = np.arange(len(order)); w = 0.26
        for j, (lab, key, cc) in enumerate([("pixels", "pixels", "#bdbdbd"),
                                            ("FractalDB-1k", f, "#e6550d"), ("ImageNet-1k", n, "#31a354")]):
            ax.bar(x + (j - 1) * w, [df.loc[key, e] for e in order], w, label=lab, color=cc)
        ax.axvline(len(IMG_ENC) - 0.5, color="k", lw=0.8, ls=":")
        ax.set_title(f"{'ResNet-50' if arch=='cnn' else 'DeiT'} — {tag}")
        ax.set_xticks(x, [e + (" *" if e in DEGENERATE else "") for e in order], rotation=30)
        if col == 0: ax.set_ylabel(f"rho ({tag})")
        if rowi == 0 and col == 1: ax.legend(fontsize=8)
fig.suptitle("De-anisotropization (held-out): a small residual fractal signal above whitened controls; "
             "natural images remain far stronger (* GPT-2 collapsed, stratified out)", y=1.01)
fig.tight_layout(); fig.savefig(OUT / "G9_whitening_rescue.png", dpi=150, bbox_inches="tight"); plt.show()

# five-view ablation figure (image-encoder means)
fig2, axes2 = plt.subplots(1, 2, figsize=(15, 4.5))
for ax, arch in zip(axes2, ARCHW):
    r, f, n = ARCHW[arch]; xv = np.arange(len(VIEWS_ORDER)); w = 0.2
    for j, (lab, key, cc) in enumerate([("pixels", "pixels", "#bdbdbd"), ("random", r, "#9ecae1"),
                                        ("fractal", f, "#e6550d"), ("ImageNet", n, "#31a354")]):
        ax.bar(xv + (j - 1.5) * w, [mean_img(RHO[v], key) for v in VIEWS_ORDER], w, label=lab, color=cc)
    ax.set_xticks(xv, VIEWS_ORDER); ax.set_title(f"{'ResNet-50' if arch=='cnn' else 'DeiT'}: image-encoder mean rho by view")
    ax.set_ylabel("mean rho over image encoders")
    if arch == "cnn": ax.legend(fontsize=8)
fig2.suptitle("Which operation makes the fractal ordering visible? (held-out)", y=1.02)
fig2.tight_layout(); fig2.savefig(OUT / "G9_ablation_views.png", dpi=150, bbox_inches="tight"); plt.show()

for v in VIEWS_ORDER:
    RHO[v].to_csv(OUT / f"G9_whitening_{v}.csv")
json.dump(dict(whiten_dim=WHITEN_DIM, verified=VERIFIED, held_out=int(len(HO)), train=int(len(TR)),
               image_mean_by_view={v: {k: mean_img(RHO[v], k) for k in BRACKET} for v in VIEWS_ORDER},
               verdicts_whitened={f"{p}_{a}": list(map(str, v)) for (p, a), v in VW.items()},
               R_white={f"{a}_{e}": (None if not np.isfinite(v) else v) for (a, e), v in Rw_tab.items()},
               cnn_signal=bool(VW[("P5w", "cnn")][0] == "CONFIRMED" and VW[("P2w", "cnn")][0] == "CONFIRMED")),
          open(OUT / "G9_whitening_rescue.json", "w"), indent=2)

# --- rewrite the PRH evidence table with the F.6 held-out diagnostic row filled ---
def _mi(df, k):
    return float(np.mean([df.loc[k, e] for e in IMG_ENC]))
p5w_cnn, p5w_vit = VW[("P5w", "cnn")][0], VW[("P5w", "vit")][0]
p2w_cnn = VW[("P2w", "cnn")][0]
cnn_signal = (p5w_cnn == "CONFIRMED" and p2w_cnn == "CONFIRMED")
g9_full = dict(g9)   # carry F.1-F.5 from cell 16
g9_full["F6"] = (STAMP + f"HELD-OUT ({len(HO)} rows), transforms fit on {len(TR)} train rows. frac_cnn "
                 f"image-mean rho by view: raw {_mi(RHO['raw'],'frac_cnn'):.3f} (pixel floor "
                 f"{_mi(RHO['raw'],'pixels'):.3f}) -> centered {_mi(RHO['centered'],'frac_cnn'):.3f} -> "
                 f"whitened {_mi(RHO['white'],'frac_cnn'):.3f} (floor {_mi(RHO['white'],'pixels'):.3f}, "
                 f"random {_mi(RHO['white'],'rand_cnn'):.3f}). Absolute rho FALLS; the ORDERING appears. "
                 f"P2w-cnn {p2w_cnn} {VW[('P2w','cnn')][1]}/4, P5w-cnn {p5w_cnn} {VW[('P5w','cnn')][1]}/4, "
                 f"P5w-vit {p5w_vit} {VW[('P5w','vit')][1]}/4. R_white image cnn {Rw_detail['cnn']['img']:.3f}, "
                 f"vit {Rw_detail['vit']['img']:.3f}")
g9_full["F6c"] = "Random twin and 32x32 pixels as floors, transformed the same way; held-out rows only; shuffle null ~0"
_size = "small" if Rw_detail["cnn"]["img"] < 0.25 else "moderate"
g9_full["F6p"] = (STAMP + (f"Raw fractal-CNN agreement was at the pixel floor because the space is highly "
                  f"anisotropic (pair-cos 0.85), not because structure is absent: after held-out "
                  f"de-anisotropization the fractal CNN reliably exceeds whitened pixel and random controls "
                  f"across all four image encoders (a {_size} residual, {Rw_detail['cnn']['img']:.0%} of the "
                  f"natural-image bracket). Natural-image twins remain far stronger. This exposes a masked "
                  f"signal; it is NOT the Atlas sense of recoverable (entry-map + frozen head) -- logged as future work."
                  if cnn_signal else
                  "Even after held-out de-anisotropization the fractal CNN does not exceed whitened pixel/random "
                  "controls: no image-encoder-shared structure at this pooling."))
TABLE_DF2 = build_table(g9_full)
TABLE_DF2.to_csv(OUT / "PRH_Evidence_Table.csv", index=False)
write_xlsx(TABLE_DF2, OUT / "PRH_Evidence_Table.xlsx", "PRH evidence table - with G9 fractal results incl. whitening rescue (F.6)")
try:
    (OUT / "PRH_Evidence_Table.md").write_text(TABLE_DF2.to_markdown(index=False))
except ImportError:
    (OUT / "PRH_Evidence_Table.md").write_text(TABLE_DF2.to_string(index=False))
print("evidence table rewritten with F.6 filled ->", (OUT / 'PRH_Evidence_Table.xlsx').name)

print("written:", *sorted(p.name for p in OUT.iterdir() if "whitening" in p.name), sep="\n  ")
print("\nRead-out: if the CNN is rescued by whitening, the raw P5-cnn 'failure' was collapse masking "
      "recoverable structure (E.6/E.12 lesson), not the absence of it. If it is NOT rescued, the fractal "
      "CNN genuinely carries no image-encoder-shared structure at this pooling. Report whichever the numbers show.")

---
## PASTE BACK FOR VERIFICATION — G9

Copy these **printed blocks** and **images** into the chat:

**Printed text:**
1. The **reproduction gate** block (`REPRODUCTION GATE ... VERIFIED`) — confirms the caches are still
   correct.
2. The **PRE-REGISTERED VERDICTS** block (P1/P2/P3/P5 with the R lines). *(P3 should now read
   INCONCLUSIVE with a real-width CI — NOT the old tight `[+0.036,+0.055]` CONFIRMED.)*
3. The **HELD-OUT WHITENED DIAGNOSTIC verdicts** block (P1w..P5w, R_white, the RESULT lines).
   *(P3w should be INCONCLUSIVE or NOT ESTIMABLE with a real-width CI, never `[+0.000,+0.000]`.)*
4. The **"Which operation moves the ordering?"** table (raw/centered/drop1/pca/white for frac_cnn
   and frac_vit). *(drop1 is now the corrected version — this is the one whose numbers changed.)*
5. The **whitening-dimension sweep** table (k = 64..512, ordering_holds column).

**Images (upload from `G9_fractal/`):**
- `G9_fractal_brackets.png`
- `G9_rho_heatmaps.png`
- `G9_ablation_views.png`   *(the five-view bars — drop1 column changed)*
- `G9_whitening_heatmap.png`
- `G9_whiten_dim_sweep.png`

If your kernel is still warm, you only need to re-run from the **verdicts cell (§14) downward** —
the extraction and ranking above did not change.